# Plan de SFT para MedGemma

Cuaderno reorganizado para entrenar, evaluar y comparar variantes de MedGemma para resúmenes PLS: prepara datos, ajusta con Unsloth/LoRA, ejecuta evaluaciones cuantitativas y cualitativas, y mantiene flujos separados para AlignScore.

## Tabla de contenidos
- [1. Instalación y configuración inicial](#1-instalacion-y-configuracion-inicial)
- [2. Carga y preprocesamiento de datos](#2-carga-y-preprocesamiento-de-datos)
- [3. Configuración del modelo y LoRA](#3-configuracion-del-modelo-y-lora)
- [4. Generación y limpieza de salidas](#4-generacion-y-limpieza-de-salidas)
- [5. Evaluación cuantitativa y preparación](#5-evaluacion-cuantitativa-y-preparacion)
- [6. Evaluación cualitativa y profiling](#6-evaluacion-cualitativa-y-profiling)
- [7. Generación con modelos fundacionales](#7-generacion-con-modelos-fundacionales)
- [8. Evaluación modular sin AlignScore](#8-evaluacion-modular-sin-alignscore)
- [9. Flujo AlignScore en entorno aislado](#9-flujo-alignscore-en-entorno-aislado)


## 1. Instalación y configuración inicial
Instala dependencias, importa librerías base, autentica con Hugging Face/Drive y define variables clave para el proyecto.

In [ ]:
# Celda 1: Instalación de librerías (Actualizada)
# MEJORA: Añadimos 'unsloth' para un fine-tuning más rápido y eficiente en memoria.
# MEJORA: Añadimos 'summac' para la métrica de factualidad y 'textstat' para legibilidad completa.
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q transformers datasets scikit-learn tensorboard
!pip install -q evaluate bert_score textstat tqdm

# NLTK ya no es necesario, pero añadimos s
!pip install -q datasets evaluate bert_score textstat tqdm pandas numpy spacy
# Descargamos el modelo de lenguaje pequeño para inglés de spaCy
!python -m spacy download en_core_web_sm

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 283.5/283.5 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.6/213.6 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.4/176.4 kB 14.5 MB/s

In [ ]:
# Celda 2: Importación de librerías (Actualizada)
import os
import torch
import gc
import time
import pandas as pd
import numpy as np
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets
from collections import defaultdict

# MEJORA: Importaciones de Unsloth
from unsloth import FastLanguageModel

from transformers import (
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig
from trl import SFTTrainer

# --- Para evaluación ---
from tqdm.auto import tqdm

# Importaciones clave (VOLVEMOS A USAR UNSLOTH)
from unsloth import FastLanguageModel
from datasets import load_dataset, Dataset, DatasetDict
import evaluate
import textstat
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification # <<<--- NUEVA IMPORTACIÓN
import spacy # <--- NUEVA IMPORTACIÓN

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
# El resto de las celdas de configuración (3, 4, 5) para autenticación y montaje de Drive se mantienen igual.
# Asegúrate de que las rutas en la Celda 5 sean correctas.
# ... (código de login y montaje de Drive) ...

In [ ]:
# Celda 3: Login en Hugging Face
from google.colab import userdata
from huggingface_hub import login

print("Configurando la API Key de HuggingFace...")
try:
    # Obtener la API key de los "Secrets" de Google Colab
    # Asegúrate de haber agregado tu HUGGING_API_KEY a los Secrets
    API_KEY = userdata.get('HUGGING_API_KEY')
    login(token=API_KEY)
    print("API Key configurada exitosamente.")
except Exception as e:
    print(f"Error al configurar la API Key: {e}")
    print("Por favor, asegúrate de que 'HUGGING_API_KEY' esté guardada en los 'Secrets' de Colab (icono de la llave).")
    raise

Configurando la API Key de HuggingFace...
API Key configurada exitosamente.


In [ ]:
# Celda 4: Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Celda 5.1: Definición de variables clave
import os
BASE_PROJECT_DIR = "/content/drive/MyDrive/NLP_Project_MedGemma"
MODEL_OUTPUT_DIR = os.path.join(BASE_PROJECT_DIR, "medgemma-finetuned-v8")
LOGS_DIR = os.path.join(BASE_PROJECT_DIR, "logs-v8")
MODEL_ID = "google/medgemma-4b-it" # Usamos la versión más reciente y ligera para empezar

os.makedirs(MODEL_OUTPUT_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

print(f"Directorio del proyecto: {BASE_PROJECT_DIR}")
print(f"Directorio de salida del modelo: {MODEL_OUTPUT_DIR}")
print(f"Modelo base a utilizar: {MODEL_ID}")

Directorio del proyecto: /content/drive/MyDrive/NLP_Project_MedGemma
Directorio de salida del modelo: /content/drive/MyDrive/NLP_Project_MedGemma/medgemma-finetuned-v8
Modelo base a utilizar: google/medgemma-4b-it


## 2. Carga y preprocesamiento de datos
Clona el repositorio de datos, combina todas las fuentes, convierte a pandas, muestra ejemplos y formatea prompts para todos los splits.

In [ ]:
# Celda 6: Clonar el repositorio con los datos
GIT_REPO_URL = "https://github.com/feliperussi/bridging-the-gap-in-health-literacy.git"
REPO_DIR = "/content/bridging-the-gap-in-health-literacy"

if not os.path.exists(REPO_DIR):
    !git clone {GIT_REPO_URL}
else:
    print("El repositorio ya ha sido clonado.")

# Definimos la ruta principal a los datos que nos interesan
DATA_SOURCES_DIR = os.path.join(REPO_DIR, "data_collection_and_processing/Data Sources")
print(f"Datos descargados en: {DATA_SOURCES_DIR}")
!ls -l "{DATA_SOURCES_DIR}"

Cloning into 'bridging-the-gap-in-health-literacy'...
remote: Enumerating objects: 72074, done.
remote: Counting objects: 100% (3/3), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 72074 (delta 0), reused 2 (delta 0), pack-reused 72071 (from 2)
Receiving objects: 100% (72074/72074), 315.90 MiB | 21.31 MiB/s, done.
Resolving deltas: 100% (2991/2991), done.
Updating files: 100% (87209/87209), done.
Datos descargados en: /content/bridging-the-gap-in-health-literacy/data_collection_and_processing/Data Sources
total 20
drwxr-xr-x 4 root root 4096 Nov 18 12:34  ClinicalTrials.gov
drwxr-xr-x 4 root root 4096 Nov 18 12:34  Cochrane
drwxr-xr-x 5 root root 4096 Nov 18 12:34  Pfizer
-rw-r--r-- 1 root root 2406 Nov 18 12:34  README.md
drwxr-xr-x 6 root root 4096 Nov 18 12:34 'Trial Summaries'


In [ ]:
# Celda 7 (Reescrita y Mejorada): Cargar, Emparejar y Estructurar TODOS los datos

import os
import pandas as pd
from collections import defaultdict
from datasets import Dataset, DatasetDict, concatenate_datasets
from tqdm.auto import tqdm # Para barras de progreso

# --- Parte 1: Funciones de Carga Específicas por Fuente ---

def load_cochrane_pairs(cochrane_dir):
    """
    Carga y empareja los textos técnicos (non_pls) y sencillos (pls)
    específicamente de la fuente Cochrane.
    Devuelve dos listas de pares: una para train y otra para test.
    """
    print(f"Procesando fuente: Cochrane desde {cochrane_dir}")
    train_pairs, test_pairs = [], []

    for split in ['train', 'test']:
        split_path = os.path.join(cochrane_dir, split)
        if not os.path.exists(split_path):
            print(f"  Advertencia: No se encontró el directorio {split_path}")
            continue

        pls_dir = os.path.join(split_path, "pls")
        non_pls_dir = os.path.join(split_path, "non_pls")

        if not os.path.exists(pls_dir) or not os.path.exists(non_pls_dir):
            print(f"  Advertencia: Faltan las carpetas 'pls' o 'non_pls' en {split_path}")
            continue

        # Usamos un diccionario para encontrar los pares eficientemente
        paired_files = defaultdict(dict)

        # 1. Procesar archivos de lenguaje sencillo (pls)
        for filename in os.listdir(pls_dir):
            # MEJORA: Aseguramos que solo procesamos el archivo principal, no las secciones
            if filename.endswith('-pls.txt'):
                base_name = filename.replace('-pls.txt', '')
                paired_files[base_name]['pls'] = os.path.join(pls_dir, filename)

        # 2. Procesar archivos de lenguaje técnico (non_pls)
        for filename in os.listdir(non_pls_dir):
            # MEJORA: Aseguramos que solo procesamos el archivo principal
            if filename.endswith('-abstract.txt'):
                base_name = filename.replace('-abstract.txt', '')
                paired_files[base_name]['non_pls'] = os.path.join(non_pls_dir, filename)

        # 3. Leer el contenido de los archivos emparejados
        target_list = train_pairs if split == 'train' else test_pairs
        for base_name, paths in tqdm(paired_files.items(), desc=f"Emparejando Cochrane {split}"):
            if 'pls' in paths and 'non_pls' in paths:
                try:
                    with open(paths['non_pls'], 'r', encoding='utf-8') as f:
                        technical_text = f.read()
                    with open(paths['pls'], 'r', encoding='utf-8') as f:
                        plain_summary = f.read()

                    if technical_text and plain_summary:
                        target_list.append({
                            "technical_text": technical_text,
                            "plain_summary": plain_summary,
                            "source": "Cochrane"
                        })
                except Exception as e:
                    print(f"  Error al leer el par para {base_name}: {e}")

    return train_pairs, test_pairs


def load_pfizer_clinicaltrials_pairs(pfizer_dir, clinicaltrials_dir):
    """
    Carga y empareja los resúmenes sencillos (pls) de Pfizer con los textos
    técnicos correspondientes de ClinicalTrials.gov usando el NCT ID.
    Devuelve dos listas de pares: una para train y otra para test.
    """
    print(f"Procesando fuentes: Pfizer y ClinicalTrials.gov")
    train_pairs, test_pairs = [], []

    for split in ['train', 'test']:
        # Ruta a los resúmenes sencillos de Pfizer
        pfizer_pls_dir = os.path.join(pfizer_dir, split, "pls")
        # Ruta a los textos técnicos de ClinicalTrials.gov
        clinicaltrials_tech_dir = os.path.join(clinicaltrials_dir, split)

        if not os.path.exists(pfizer_pls_dir) or not os.path.exists(clinicaltrials_tech_dir):
            print(f"  Advertencia: Faltan directorios para el split '{split}' en Pfizer o ClinicalTrials.gov")
            continue

        target_list = train_pairs if split == 'train' else test_pairs
        for filename in tqdm(os.listdir(pfizer_pls_dir), desc=f"Emparejando Pfizer/ClinicalTrials {split}"):
            # MEJORA: Nos aseguramos de tomar el archivo base (NCT...txt) y no sus secciones
            if filename.startswith('NCT') and filename.endswith('.txt') and '_section' not in filename:
                nct_id = filename

                # Buscar el archivo técnico correspondiente en ClinicalTrials.gov
                tech_file_path = os.path.join(clinicaltrials_tech_dir, nct_id)

                if os.path.exists(tech_file_path):
                    try:
                        with open(tech_file_path, 'r', encoding='utf-8') as f:
                            technical_text = f.read()
                        with open(os.path.join(pfizer_pls_dir, filename), 'r', encoding='utf-8') as f:
                            plain_summary = f.read()

                        if technical_text and plain_summary:
                            target_list.append({
                                "technical_text": technical_text,
                                "plain_summary": plain_summary,
                                "source": "Pfizer/ClinicalTrials"
                            })
                    except Exception as e:
                        print(f"  Error al leer el par para {nct_id}: {e}")

    return train_pairs, test_pairs

# --- Parte 2: Orquestador de Carga de Datos ---

DATA_SOURCES_DIR = "/content/bridging-the-gap-in-health-literacy/data_collection_and_processing/Data Sources"
cochrane_train, cochrane_test = load_cochrane_pairs(os.path.join(DATA_SOURCES_DIR, "Cochrane"))
pfizer_train, pfizer_test = load_pfizer_clinicaltrials_pairs(
    os.path.join(DATA_SOURCES_DIR, "Pfizer"),
    os.path.join(DATA_SOURCES_DIR, "ClinicalTrials.gov")
)

# Nota sobre "Trial Summaries":
print("\nNota: La fuente 'Trial Summaries' se omite en esta fase porque no contiene pares de textos (técnico/sencillo) claramente estructurados, solo resúmenes en lenguaje sencillo.")

all_train_pairs = cochrane_train + pfizer_train
all_test_pairs = cochrane_test + pfizer_test

print(f"\nTotal de pares de entrenamiento combinados: {len(all_train_pairs)}")
print(f"Total de pares de prueba combinados: {len(all_test_pairs)}")

if all_train_pairs and all_test_pairs:
    train_ds = Dataset.from_pandas(pd.DataFrame(all_train_pairs))
    test_ds = Dataset.from_pandas(pd.DataFrame(all_test_pairs))
    train_val_split = train_ds.train_test_split(test_size=0.1, seed=42)

    final_dataset = DatasetDict({
        'train': train_val_split['train'],
        'validation': train_val_split['test'],
        'test': test_ds
    })

    print("\n--- Distribución Final del Dataset Combinado ---")
    print(f"Entrenamiento: {len(final_dataset['train'])} ejemplos")
    print(f"Validación:    {len(final_dataset['validation'])} ejemplos")
    print(f"Prueba:        {len(final_dataset['test'])} ejemplos")

    # AQUÍ SE DEFINEN LAS VARIABLES QUE CAUSABAN EL ERROR
    train_dataset = final_dataset['train']
    val_dataset = final_dataset['validation']
    test_dataset = final_dataset['test']
else:
    raise RuntimeError("No se cargaron datos. Revisa las rutas y la estructura de archivos.")


Procesando fuente: Cochrane desde /content/bridging-the-gap-in-health-literacy/data_collection_and_processing/Data Sources/Cochrane


Emparejando Cochrane train:   0%|          | 0/7559 [00:00<?, ?it/s]

Emparejando Cochrane test:   0%|          | 0/2567 [00:00<?, ?it/s]

Procesando fuentes: Pfizer y ClinicalTrials.gov


Emparejando Pfizer/ClinicalTrials train:   0%|          | 0/490 [00:00<?, ?it/s]

Emparejando Pfizer/ClinicalTrials test:   0%|          | 0/116 [00:00<?, ?it/s]


Nota: La fuente 'Trial Summaries' se omite en esta fase porque no contiene pares de textos (técnico/sencillo) claramente estructurados, solo resúmenes en lenguaje sencillo.

Total de pares de entrenamiento combinados: 3641
Total de pares de prueba combinados: 221

--- Distribución Final del Dataset Combinado ---
Entrenamiento: 3276 ejemplos
Validación:    365 ejemplos
Prueba:        221 ejemplos


In [ ]:
# Convertir final_dataset (type DatasetDict) a df de pandas
import pandas as pd
import os

# Convert each split to a pandas DataFrame and concatenate them
df_train = final_dataset['train'].to_pandas()
df_validation = final_dataset['validation'].to_pandas()
df_test = final_dataset['test'].to_pandas()

# Guardar el final_dataset como csv
BASE_DATASETS_DIR = os.path.join(BASE_PROJECT_DIR, "medgemma-finetuned-v8_datasets")

os.makedirs(BASE_DATASETS_DIR, exist_ok=True)

df_train_path = os.path.join(BASE_DATASETS_DIR, 'df_train.csv')
df_validation_path = os.path.join(BASE_DATASETS_DIR, 'df_validation.csv')
df_test_path = os.path.join(BASE_DATASETS_DIR, 'df_test.csv')
df_train.to_csv(df_train_path, index=False)
df_validation.to_csv(df_validation_path, index=False)
df_test.to_csv(df_test_path, index=False)

print(f"Datasets guardados en: {df_train_path, df_validation_path, df_test_path}")

Datasets guardados en: ('/content/drive/MyDrive/NLP_Project_MedGemma/medgemma-finetuned-v8_datasets/df_train.csv', '/content/drive/MyDrive/NLP_Project_MedGemma/medgemma-finetuned-v8_datasets/df_validation.csv', '/content/drive/MyDrive/NLP_Project_MedGemma/medgemma-finetuned-v8_datasets/df_test.csv')


In [ ]:
# Celda 7.1: Imprimir un par de ejemplo del dataset combinado
if final_dataset and len(final_dataset['train']) > 0:
    print("\n--- Ejemplo de un par de texto técnico y resumen en lenguaje sencillo ---")
    example = final_dataset['validation'][220] # Tomamos el primer ejemplo del conjunto de entrenamiento
    print("\nTexto Técnico:")
    print(example['technical_text'][::] + "...") # Imprimimos solo los primeros 500 caracteres
    print("\nResumen en Lenguaje Sencillo (PLS):")
    print(example['plain_summary'][:] + "...") # Imprimimos solo los primeros 500 caracteres
    print("\nFuente:", example['source'])
else:
    print("\nEl dataset final está vacío o no ha sido creado correctamente.")


--- Ejemplo de un par de texto técnico y resumen en lenguaje sencillo ---

Texto Técnico:
Background
Active management of the third stage of labour reduces the risk of postpartum blood loss (postpartum haemorrhage (PPH)), and is defined as administration of a prophylactic uterotonic, early umbilical cord clamping and controlled cord traction to facilitate placental delivery. The choice of uterotonic varies across the globe and may have an impact on maternal outcomes. This is an update of a review first published in 2001 and last updated in 2013. 
Objectives
To determine the effectiveness of prophylactic oxytocin to prevent PPH and other adverse maternal outcomes in the third stage of labour. 
Search methods
For this update, we searched Cochrane Pregnancy and Childbirth’s Trials Register, ClinicalTrials.gov, WHO International Clinical Trials Registry Platform (ICTRP) (6 March 2019) and reference lists of retrieved studies. 
Selection criteria
Randomised, quasi‐ or cluster‐randomised tr

In [ ]:
# Celda 8 (Reescrita y Mejorada): Definición del Prompt y Preprocesamiento para Todos los Splits

# MEJORA: Reemplazamos el prompt simple por la plantilla avanzada y optimizada en español.
# Esta plantilla establece un rol, define reglas estrictas y alinea la tarea con las métricas.
def create_optimized_prompt(technical_text, plain_summary=None):
    """
    Crea la plantilla de prompt completa. Si se proporciona un 'plain_summary',
    se genera el formato para entrenamiento (SFT). De lo contrario, se genera
    el formato para inferencia.
    """
    system_prompt = (
        "You are a Health Literacy Expert."
        "Your expertise is in rewriting complex, technical medical texts into clear, simple, and accurate language for a general audience, following established health communication guidelines."
    )

    instruction_template = f"""**[PRIMARY GOAL]**
        Your main purpose is to rewrite the provided medical text into a Plain Language Summary (PLS).
        This summary must be easy to understand for someone with an 8th-grade reading comprehension level typical of a general middle-school student,
        consistent with plain-language standards used by CDC and NIH, meaning it should use simple vocabulary, short sentences,
        and concepts that can be understood by someone with basic middle-school literacy,
        while remaining completely faithful to the source's essential information, for intance, conclusions.

        **[TASK INSTRUCTION]**
        Rewrite the following technical medical text into a Plain Language Summary (PLS).
        The output **MUST BE ONLY** the plain language summary without special symbols and stop tokens.

        **--- STRICT OUTPUT RULES ---**

        1.  **ACCURACY AND COMPLETENESS:**
        *   The summary MUST retain all key findings, main outcomes, important safety information, conclusions,and any significant numerical results from the original text.
        *   Do NOT add any information, opinions, or recommendations that are not present in the source document. The summary must be based ONLY on the provided text.

        2.  **CLARITY AND READABILITY:**
        *   Write the summary at an **8th-grade reading comprehension level typical of a general middle-school student**.
        *   Use short, clear, and natural-sounding sentences.
        *   Use the active voice whenever possible (e.g., "Scientists tested the drug" instead of "The drug was tested by scientists").
        *   Avoid long, complex words when a simpler alternative exists.

        3.  **TERMINOLOGY (JARGON):**
        *   Avoid medical jargon.
        *   If a technical term is absolutely essential and cannot be replaced, you MUST explain it simply in parentheses the first time it appears. (Example: "The trial used immunotherapy (a treatment that helps the body's immune system fight cancer).")

        4.  **FORMATTING AND LANGUAGE:**
        *   The output **MUST BE ONLY** the plain language summary without special symbols and stop tokens, **only the summary**.
        *   The output must be written **ONLY in English**.
        *   Structure the summary as a set of concise paragraphs. Use as many sentences as needed to include all essential information, up to a maximum length of about 500 words, however, **make sure ALL sentences are complete, which means ALL ideas are finished.**
        *   Do NOT include headings, bullet points, lists, citations, or URLs.

        **--- SOURCE TECHNICAL TEXT ---**
        <document>
        {technical_text}
        </document>

        **--- PLAIN LANGUAGE SUMMARY (PLS) ---**
      """

    if plain_summary is not None:
        # === CAMBIO CLAVE PARA ENTRENAMIENTO ===
        # El formato ahora es: INSTRUCCIÓN -> RESPUESTA -> FIN DE SECUENCIA
        # Eliminamos tus etiquetas personalizadas <plain_language_summary>.
        # La respuesta (plain_summary) va directamente después de [/INST].
        # Añadimos el token de fin de secuencia (</s>) al final de la respuesta.
        # Esto enseña al modelo que, cuando termine de generar el resumen, debe emitir la señal de parada.
        prompt = f"<s>[INST] {system_prompt}\n\n{instruction_template} [/INST]\n{plain_summary}</s>"
        return prompt
    else:
        # === CAMBIO CLAVE PARA INFERENCIA ===
        # El prompt para la inferencia debe terminar EXACTAMENTE donde el modelo debe empezar a escribir.
        # Es decir, justo después de [/INST] y un salto de línea.
        prompt = f"<s>[INST] {system_prompt}\n\n{instruction_template} [/INST]\n"
        return prompt


# MEJORA: Esta función prepara los datos para el SFTTrainer, que espera una única columna de texto.
def formatting_prompts_func(example):
    # Aseguramos que los campos no sean nulos
    technical_text = example.get("technical_text", "")
    plain_summary = example.get("plain_summary", "")

    # Creamos el prompt formateado para el entrenamiento
    example["text"] = create_optimized_prompt(technical_text, plain_summary)
    return example

In [ ]:
# MEJORA: Aplicamos el formateo a TODOS los splits del dataset (train, validation, y test).
# Usamos `remove_columns` para dejar solo la columna 'text' que necesita el SFTTrainer.
print("Formateando los datasets para el entrenamiento...")

train_dataset_formatted = train_dataset.map(
    formatting_prompts_func,
    remove_columns=list(train_dataset.features)
)
val_dataset_formatted = val_dataset.map(
    formatting_prompts_func,
    remove_columns=list(val_dataset.features)
)
test_dataset_formatted = test_dataset.map(
    formatting_prompts_func,
    remove_columns=list(test_dataset.features)
)

print("\nFormateo completado.")
print(f"Columnas del dataset de entrenamiento: {train_dataset_formatted.column_names}")

# --- Verificación ---
print("\n--- Ejemplo de Prompt Formateado para Entrenamiento ---")
print(train_dataset_formatted[0]['text'])
print("-" * 50)

Formateando los datasets para el entrenamiento...


Map:   0%|          | 0/3276 [00:00<?, ? examples/s]

Map:   0%|          | 0/365 [00:00<?, ? examples/s]

Map:   0%|          | 0/221 [00:00<?, ? examples/s]


Formateo completado.
Columnas del dataset de entrenamiento: ['text']

--- Ejemplo de Prompt Formateado para Entrenamiento ---
<s>[INST] You are a Health Literacy Expert.Your expertise is in rewriting complex, technical medical texts into clear, simple, and accurate language for a general audience, following established health communication guidelines.

**[PRIMARY GOAL]**
        Your main purpose is to rewrite the provided medical text into a Plain Language Summary (PLS).
        This summary must be easy to understand for someone with an 8th-grade reading comprehension level typical of a general middle-school student,
        consistent with plain-language standards used by CDC and NIH, meaning it should use simple vocabulary, short sentences,
        and concepts that can be understood by someone with basic middle-school literacy,
        while remaining completely faithful to the source's essential information, for intance, conclusions.

        **[TASK INSTRUCTION]**
        Rewri

## 3. Configuración del modelo y LoRA
Carga el modelo con FastLanguageModel, aplica cuantización en 4 bits y configura los parámetros LoRA para el ajuste fino.

In [ ]:
# Celda 10: Configuración de Cuantización y Carga del Modelo con Unsloth
# MEJORA: Cargamos el modelo usando FastLanguageModel de Unsloth.
# Esto aplica automáticamente optimizaciones como Flash Attention.
max_seq_length = 2048 # Aumentamos un poco por el prompt más largo

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_ID,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

==((====))==  Unsloth 2025.11.3: Fast Gemma3 patching. Transformers: 4.57.1.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Gemma3 does not support SDPA - switching to fast eager.


model.safetensors:   0%|          | 0.00/4.12G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.69M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/670 [00:00<?, ?B/s]

In [ ]:
# Celda 11: Configuración de LoRA (con Mejoras)
# MEJORA: Habilitamos el entrenamiento para más capas (FFN) como recomendaste.
# Esto da más flexibilidad al modelo para adaptarse.
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, #
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      #"gate_proj", "up_proj", "down_proj"   # Capas de Atención + FFN
                      ],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = True,
    random_state = 42,
    max_seq_length = max_seq_length,
)

model.print_trainable_parameters()

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Unsloth: Making `base_model.model.model.vision_tower.vision_model` require gradients
trainable params: 11,898,880 || all params: 4,311,978,352 || trainable%: 0.2759


In [ ]:
%%time
# Celda 13 y 14: Argumentos y Ejecución del Trainer (Actualizado para Unsloth)

# --- ARGUMENTOS Y EJECUCIÓN DEL TRAINER ---

# MEJORA: Usamos SFTTrainer de TRL, que está optimizado para este tipo de tareas.
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset_formatted,
    eval_dataset = val_dataset_formatted,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Mantener en False para esta tarea de secuencia a secuencia
    args = TrainingArguments(
        # --- Parámetros de Batch y Épocas ---
        per_device_train_batch_size = 32,
        gradient_accumulation_steps = 1, # Batch efectivo de 16
        num_train_epochs = 2,

        # --- Optimizador y Scheduler ---
        learning_rate = 5e-5,
        lr_scheduler_type = "cosine",
        warmup_steps = 25,
        optim = "adamw_8bit",
        weight_decay = 0.05,

        # --- Configuración de Precisión ---
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),

        # --- Guardado, Logging y Evaluación (CORREGIDO) ---
        output_dir = MODEL_OUTPUT_DIR,
        logging_steps = 10,
        eval_strategy = "steps",  # AÑADIR/MODIFICAR: Estrategia de evaluación por pasos
        # CORRECCIÓN: Se eliminan 'evaluation_strategy' y 'save_strategy'.
        # Ahora controlamos todo explícitamente con pasos.
        save_strategy = "steps",        # AÑADIR/MODIFICAR: Estrategia de guardado por pasos
        save_steps = 30,  # Guardar un checkpoint cada 100 pasos
        eval_steps = 30,  # Evaluar en el set de validación cada 100 pasos
        save_total_limit = 3, # Guardar solo los 3 mejores checkpoints
        load_best_model_at_end = True, # Cargar el mejor modelo al final del entrenamiento

        # --- Otros ---
        seed = 42,
    ),
)

# --- Iniciar entrenamiento ---
print("\n--- Iniciando Fine-tuning del Modelo ---")
trainer_stats = trainer.train()

# --- Guardar modelo final ---
print(f"\nGuardando el mejor modelo en: {MODEL_OUTPUT_DIR}")
trainer.save_model() # Guarda el mejor adaptador en el output_dir

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/3276 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/365 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.



--- Iniciando Fine-tuning del Modelo ---


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,276 | Num Epochs = 2 | Total steps = 206
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 1 x 1) = 32
 "-____-"     Trainable parameters = 11,898,880 of 4,311,978,352 (0.28% trained)
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: rich-coding (rich-coding-uniandes) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss,Validation Loss
30,1.754800,1.647575
60,0.929300,0.800586
90,0.724100,0.721814
120,0.711500,0.707908
150,0.698900,0.700804
180,0.698000,0.697687


Unsloth: Will smartly offload gradients to save VRAM!

Guardando el mejor modelo en: /content/drive/MyDrive/NLP_Project_MedGemma/medgemma-finetuned-v8
CPU times: user 1h 26s, sys: 43min 18s, total: 1h 43min 45s
Wall time: 1h 44min 34s


## 4. Generación y limpieza de salidas
Libera memoria, reimporta utilidades, define funciones de generación y limpieza de texto, ejecuta pruebas unitarias y vacía caché de GPU.

In [ ]:
#import torch
#import gc
##
### Eliminar las variables grandes que ya no necesitamos
### Esto le dice a Python que ya no las usamos
##del model
##del trainer
##del data_collator
##
### Forzar al recolector de basura de Python a limpiar
#gc.collect()
#
### Vaciar la caché de memoria de la GPU de PyTorch
### Este es el paso más importante para liberar la VRAM
#torch.cuda.empty_cache()

In [ ]:
%%time
import unsloth # IMPORTANTE! Debe ir al principio
import torch
import pandas as pd
import os
import textwrap
from transformers import AutoTokenizer
from unsloth import FastLanguageModel

# ==============================================================================
# PASO 1: CARGAR EL MODELO RÁPIDO Y FUSIONADO
# ==============================================================================
print("Cargando el modelo y fusionando los adaptadores para máxima velocidad...")

# Cargamos el modelo desde el checkpoint de Unsloth.
# Unsloth cargará el modelo base y aplicará los adaptadores LoRA.
max_seq_length = 4096 # Necesario como variable global
model, _ = FastLanguageModel.from_pretrained(
    model_name = MODEL_OUTPUT_DIR,
    max_seq_length = max_seq_length,
    dtype = None, # Permite a Unsloth elegir el mejor dtype
    load_in_4bit = True,
)

## --- PASO 1.1: Cargar el TOKENIZER original del modelo BASE ---
## Esta es la corrección clave. Ignoramos el tokenizer del checkpoint
## y cargamos el que sabemos que es 100% compatible con la arquitectura gemma3.
print("\nCargando el tokenizer original del modelo base para asegurar compatibilidad...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID) # MODEL_ID es "google/medgemma-4b-it"

# --- PASO 1.2: CONFIGURAR EL TOKENIZER PARA GENERACIÓN ---
# Es VITAL para evitar errores. Aunque lo recarguemos, debemos asegurarnos de esto.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left" # Esencial para la generación

# ¡¡EL TRUCO PARA LA VELOCIDAD!!
# Fusiona los adaptadores LoRA en el modelo base.
# Esto crea un modelo denso en memoria que es órdenes de magnitud más rápido para la inferencia.
# Ya no es un PeftModel, es un modelo de Transformers normal pero con tu fine-tuning.
print("\nFusionando adaptadores LoRA para acelerar la inferencia...")
model.merge_and_unload()

# Ponemos el modelo en modo de evaluación
model.eval()
print("¡Modelo fusionado y listo para inferencia a máxima velocidad!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Cargando el modelo y fusionando los adaptadores para máxima velocidad...
==((====))==  Unsloth 2025.11.3: Fast Gemma3 patching. Transformers: 4.57.1.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Gemma3 does not support SDPA - switching to fast eager.


model.safetensors:   0%|          | 0.00/4.12G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/210 [00:00<?, ?B/s]


Cargando el tokenizer original del modelo base para asegurar compatibilidad...


tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/1.53k [00:00<?, ?B/s]


Fusionando adaptadores LoRA para acelerar la inferencia...


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:348: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


¡Modelo fusionado y listo para inferencia a máxima velocidad!
CPU times: user 1min 29s, sys: 20.4 s, total: 1min 49s
Wall time: 2min 7s


In [ ]:
# Función para la generación de PLS (CORREGIDA)
def generate_summary_unsloth(model, tokenizer, technical_text):
    prompt = create_optimized_prompt(technical_text)
    inputs = tokenizer(prompt,
                       return_tensors="pt",
                       truncation=True,
                       max_length=max_seq_length
                       ).to("cuda")

    # Guardamos la longitud del input para poder separarlo del output después
    input_ids_length = inputs.input_ids.shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=800,
            do_sample=True,
            top_p=0.9,
            temperature=0.3,
            # === CAMBIO CLAVE: CONDICIÓN DE PARADA SIMPLIFICADA ===
            # Ahora solo necesitamos el token oficial de fin de secuencia (EOS).
            # El fine-tuning le enseñará al modelo a generar este token al final.
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    # === CAMBIO CLAVE: DECODIFICACIÓN LIMPIA ===
    # 1. Seleccionamos solo los tokens generados (los que van después del prompt inicial).
    generated_token_ids = outputs[0, input_ids_length:]
    # 2. Decodificamos únicamente esa parte.
    decoded_output = tokenizer.decode(generated_token_ids, skip_special_tokens=True)

    try:
        # Retornar TODO lo generado por el LLM para posterior limpieza y comparación
        return decoded_output.strip(), outputs
    except IndexError as e:
        print(f"Advertencia: No se pudo retornar la salida del LLM: {e}")
        return decoded_output.strip(), outputs # Devolver lo que se haya generado después de la instrucción
    except Exception as e:
        print(f"Error inesperado: {e}")
        return decoded_output.strip(), outputs # Devolver lo que se haya generado después de la instrucción

In [ ]:
%%time
import pandas as pd
import os
import textwrap
import re  # <--- IMPORTANTE: Importar regex

# ==============================================================================
# FUNCIÓN DE LIMPIEZA
# ==============================================================================
def clean_llm_response(text):
    # Agrega aquí nuevos patrones si el modelo alucina cosas nuevas
    stop_patterns = [
        r"\[/INST\]",
        r"\[INST\]",
        r"</plain-language-summary>",
        r"<plain-language-summary>",
        r"\[/SUMMARY\]",
        r"\[/DOCUMENT\]",
        r"\[/PLAIN_LANGUAGE_SUMMARY\]",
        r"\[PLAIN_LANGUAGE_SUMMARY\]",
    ]

    combined_pattern = "|".join(stop_patterns)
    match = re.search(combined_pattern, text, re.IGNORECASE)

    if match:
        return text[:match.start()].strip()
    return text.strip()

CPU times: user 8 µs, sys: 0 ns, total: 8 µs
Wall time: 12.4 µs


In [ ]:
%%time
# ==============================================================================
# PASO 3: EJECUCIÓN PRUEBA UNITARIA
# ==============================================================================
print("\n--- Ejecutando prueba unitaria con limpieza de patrones ---")

input_csv_path = os.path.join(BASE_PROJECT_DIR, 'evaluation_texts.csv')

try:
    df_eval = pd.read_csv(input_csv_path)
    random_sample = df_eval.iloc[50]
    technical_text_sample = random_sample['technical_text']
    reference_summary_sample = random_sample['reference_summary']

    # Generar resumen (Raw output)
    raw_summary, complete = generate_summary_unsloth(model, tokenizer, technical_text_sample)

    # --- APLICAR LIMPIEZA AQUÍ ---
    # Procesamos el texto crudo para quitar los artefactos y bucles
    final_summary_clean = clean_llm_response(raw_summary)

    # --- Visualización ---
    wrapper = textwrap.TextWrapper(width=80, initial_indent="    ", subsequent_indent="    ")

    print("\n\033[1m" + "1. TEXTO TÉCNICO ORIGINAL:" + "\033[0m")
    print(wrapper.fill(technical_text_sample))
    print("\n" + "="*50 + "\n")

    print("\033[1m" + "2. RESUMEN DE REFERENCIA (Humano):" + "\033[0m")
    print("\033[92m" + wrapper.fill(reference_summary_sample) + "\033[0m")
    print("\n" + "="*50 + "\n")

    print("\033[1m" + "3. RESUMEN GENERADO (Modelo Fine-Tuned + Regex Clean):" + "\033[0m")
    # Imprimimos la versión limpia
    print("\033[94m" + wrapper.fill(final_summary_clean) + "\033[0m")

except NameError as e:
    print(f"ERROR: {e}")
except Exception as e:
    print(f"Ocurrió un error inesperado: {e}")


--- Ejecutando prueba unitaria con limpieza de patrones ---

1. TEXTO TÉCNICO ORIGINAL:
    Background Community‐based primary‐level workers (PWs) are an important
    strategy for addressing gaps in mental health service delivery in low‐ and
    middle‐income countries.   Objectives To evaluate the effectiveness of
    PW‐led treatments for persons with mental health symptoms in LMICs, compared
    to usual care.   Search methods MEDLINE, Embase, CENTRAL,
    ClinicalTrials.gov, ICTRP, reference lists (to 20 June 2019).   Selection
    criteria Randomised trials of PW‐led or collaborative‐care interventions
    treating people with mental health symptoms or their carers in LMICs.   PWs
    included: primary health professionals (PHPs), lay health workers (LHWs),
    community non‐health professionals (CPs).   Data collection and analysis
    Seven conditions were identified apriori and analysed by disorder
    and PW examining recovery, prevalence, symptom change, quality‐of‐life
   

In [ ]:
# Analizando la salida completa vs la limpiada
decoded_output = tokenizer.decode(complete[0], skip_special_tokens=True)
decoded_output

'<s>[INST] You are a Health Literacy Expert.Your expertise is in rewriting complex, technical medical texts into clear, simple, and accurate language for a general audience, following established health communication guidelines.\n\n**[PRIMARY GOAL]**\n        Your main purpose is to rewrite the provided medical text into a Plain Language Summary (PLS).\n        This summary must be easy to understand for someone with an 8th-grade reading comprehension level typical of a general middle-school student,\n        consistent with plain-language standards used by CDC and NIH, meaning it should use simple vocabulary, short sentences,\n        and concepts that can be understood by someone with basic middle-school literacy,\n        while remaining completely faithful to the source\'s essential information, for intance, conclusions.\n\n        **[TASK INSTRUCTION]**\n        Rewrite the following technical medical text into a Plain Language Summary (PLS).\n        The output **MUST BE ONLY** t

In [ ]:
# limpia gpu ram
import torch
torch.cuda.empty_cache()

## 5. Evaluación cuantitativa y preparación
Configura la evaluación, carga resultados previos, calcula métricas, prepara un entorno separado para AlignScore y genera scripts auxiliares.

In [ ]:
import torch
import pandas as pd
import os
import re
from tqdm.auto import tqdm

# ==============================================================================
# 1. CONFIGURACIÓN
# ==============================================================================
BATCH_SIZE = 12   # Ajusta según tu VRAM
MAX_SEQ_LEN = 4096 # Tu longitud máxima configurada en el modelo

# ==============================================================================
# 2. FUNCIONES AUXILIARES
# ==============================================================================

def clean_llm_response(text):
    """
    Limpia el texto asumiendo que recibe SOLO la respuesta generada (sin prompt).
    Corta en la primera señal de parada encontrada.
    """
    if not isinstance(text, str): return ""

    stop_patterns = [
        r"\[/INST\]", r"\[INST\]",
        r"</plain-language-summary>", r"<plain-language-summary>",
        r"\[/SUMMARY\]", r"\[/DOCUMENT\]",
        r"\[/PLAIN_LANGUAGE_SUMMARY\]", r"\[PLAIN_LANGUAGE_SUMMARY\]",
        r"<<",
    ]
    combined_pattern = "|".join(stop_patterns)
    match = re.search(combined_pattern, text, re.IGNORECASE)

    if match:
        return text[:match.start()].strip()

    return text.strip()

def generate_batch_debug(model, tokenizer, texts_batch):
    """
    Genera texto y separa matemáticamente el Prompt de la Respuesta usando tensores,
    evitando errores de string splitting.
    """
    prompts = [create_optimized_prompt(t) for t in texts_batch]

    # Tokenizamos con padding a la IZQUIERDA (vital para decoder-only models)
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LEN
    ).to("cuda")

    # Guardamos la longitud exacta de la entrada (incluyendo padding)
    input_length = inputs.input_ids.shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=800,
            do_sample=True,
            top_p=0.9,
            temperature=0.3,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id,
            use_cache=True
        )

    # --- AQUÍ ESTÁ LA MAGIA DE LA SEPARACIÓN ---

    # 1. Full Text: Decodificamos todo el tensor (Prompt + Respuesta + Padding final si lo hubiera)
    full_texts = tokenizer.batch_decode(outputs, skip_special_tokens=False)

    # 2. Response Only: Cortamos el tensor para quedarnos SOLO con los tokens nuevos
    #    outputs[:, input_length:] toma todas las filas, desde la columna input_length hasta el final.
    generated_tokens = outputs[:, input_length:]
    raw_responses = tokenizer.batch_decode(generated_tokens, skip_special_tokens=False)

    return full_texts, raw_responses

# ==============================================================================
# 3. PROCESAMIENTO PRINCIPAL (MODO DEPURACIÓN)
# ==============================================================================

# Definir rutas
input_csv_path = os.path.join(BASE_PROJECT_DIR, 'evaluation_texts.csv')
output_csv_path = os.path.join(BASE_PROJECT_DIR, 'v8_debug_audit_summaries.csv')

try:
    df_eval = pd.read_csv(input_csv_path)
    # Tomamos 10 filas para la prueba rápida
    #df_to_process = df_eval.head(10).copy()
    df_to_process = df_eval.copy()
    print(f"Procesando {len(df_to_process)} filas de prueba...")
except FileNotFoundError:
    print("No se encontró el archivo CSV.")
    exit()

print(f"\n--- Iniciando generación DEBUG (Auditando {df_to_process.shape[0]} filas) ---")
print(f"\n--- Tamaño del batch: {BATCH_SIZE}")

# Listas para almacenar las 3 versiones
list_full_text_debug = []
list_raw_response_debug = []
list_clean_summary = []

technical_texts = df_to_process['technical_text'].tolist()

# --- BUCLE POR LOTES ---
for i in tqdm(range(0, len(technical_texts), BATCH_SIZE), desc="Auditando"):
    batch_texts = technical_texts[i : i + BATCH_SIZE]

    try:
        # Obtenemos Full Text y Raw Response separados por tokens, no por texto
        full_batch, raw_batch = generate_batch_debug(model, tokenizer, batch_texts)

        # Procesamos cada elemento del lote
        for full, raw in zip(full_batch, raw_batch):

            # 1. Guardar Full Text (con prompt y todo)
            list_full_text_debug.append(full)

            # 2. Guardar Raw Response (solo lo generado, sucio)
            list_raw_response_debug.append(raw)

            # 3. Limpiar y guardar Clean Summary
            clean = clean_llm_response(raw)
            list_clean_summary.append(clean)

    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            print(f"\nERROR OOM en índice {i}. Reduce BATCH_SIZE.")
            torch.cuda.empty_cache()
            break
        else:
            print(f"Error: {e}")

# --- GUARDADO DEL CSV CON TODAS LAS COLUMNAS ---
limit = len(list_clean_summary)
df_final = df_to_process.iloc[:limit].copy()

# Agregamos las columnas en orden lógico de proceso
df_final['debug_full_text'] = list_full_text_debug       # El prompt + salida
df_final['debug_raw_response'] = list_raw_response_debug # Solo salida (sucia)
df_final['generated_summary'] = list_clean_summary       # Salida limpia final

df_final.to_csv(output_csv_path, index=False, encoding='utf-8')

print("\n--- Auditoría completada ---")
print(f"Archivo guardado en: {output_csv_path}")
print("Columnas generadas: ['debug_full_text', 'debug_raw_response', 'generated_summary']")

# Visualización rápida de un caso (si existe)
if len(df_final) > 0:
    print("\nEjemplo del primer registro (Raw Response vs Clean):")
    print("-" * 30)
    print("RAW (Sucio):")
    print(df_final.iloc[0]['debug_raw_response'][:200] + "...")
    print("-" * 30)
    print("CLEAN (Limpio):")
    print(df_final.iloc[0]['generated_summary'][:200] + "...")


--- Iniciando generación en lote (Batch Size: 12) ---
Total filas a procesar: 100


Procesando Lotes:   0%|          | 0/9 [00:00<?, ?it/s]


--- Proceso completado ---
Resultados guardados en: '/content/drive/MyDrive/NLP_Project_MedGemma/v8_generated_summaries_fast.csv'
                                   reference_summary  \
0  Non‐surgical treatment for spinal stenosis wit...   
1  Interventions for involving older patients wit...   
2  Beta‐blockers for children with congestive hea...   
3  Removal of the spleen in people with thalassae...   
4  One, two or three times a week iron supplement...   

                                   generated_summary  
0                                                     
1  Older adults with many health problems (like h...  
2  The plain language summary is excellent. It ac...  
3  The Plain Language Summary (PLS) is accurate, ...  
4                                                     


In [ ]:
import pandas as pd
#from datasets import load_dataset, Dataset, DatasetDict
import evaluate
import os
import torch
import gc
import time
import pandas as pd
import numpy as np
# --- Para evaluación ---
from tqdm.auto import tqdm
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification # <<<--- NUEVA IMPORTACIÓN
import textstat
import spacy # <--- NUEVA IMPORTACIÓN

In [ ]:
# --- Celda 3: Cargar los Resultados Guardados desde CSV ---

# Definir la ruta al archivo
csv_path = os.path.join(BASE_PROJECT_DIR, 'v8_debug_audit_summaries.csv')

print(f"Cargando resultados pre-generados desde: {csv_path}")

try:
    results_df = pd.read_csv(csv_path)
    # For testing only 10
    #results_df = results_df.head(10)
except FileNotFoundError:
    print(f"ERROR: No se encontró el archivo en la ruta especificada. Asegúrate de que generaste el archivo CSV primero.")
    raise

# Limpiar datos por si acaso (eliminar filas donde la generación pudo haber fallado)
results_df.dropna(subset=['technical_text', 'reference_summary', 'generated_summary'], inplace=True)
results_df = results_df[results_df['generated_summary'].str.strip() != '']

# Extraer las listas necesarias para las métricas
clean_sources = results_df['technical_text'].tolist()
clean_references = results_df['reference_summary'].tolist()
clean_predictions = results_df['generated_summary'].tolist()

print(f"Cargados y listos para evaluar {len(clean_predictions)} resúmenes.")

Cargando resultados pre-generados desde: /content/drive/MyDrive/NLP_Project_MedGemma/qwen_pls.csv
Cargados y listos para evaluar 100 resúmenes.


In [ ]:
# <<<--- INICIO DE LA CORRECCIÓN --->>>
# Cargar el modelo de spaCy para la tokenización de oraciones
print("Cargando modelo de lenguaje de spaCy...")
nlp = spacy.load("en_core_web_sm")
# <<<--- FIN DE LA CORRECCIÓN --->>>

Cargando modelo de lenguaje de spaCy...


In [ ]:
# --- Celda 3 (Modificada): Cálculo Completo de Métricas (Modelo y Referencia) ---

# Asegurarse de tener el modelo de spacy cargado
try:
    nlp = spacy.load("en_core_web_sm")
except:
    import spacy.cli
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

print("\n--- Calculando métricas finales (Modelo vs. Referencia) ---")
device = 0 if torch.cuda.is_available() else -1
device_str = "cuda" if torch.cuda.is_available() else "cpu"

# --- 1. Relevancia Semántica (BERTScore) ---
print("Calculando BERTScore...")
bertscore_metric = evaluate.load("bertscore")

# A. Para el Modelo (Generado vs Referencia)
print("-> BERTScore: Modelo...")
bertscore_results_gen = bertscore_metric.compute(
    predictions=clean_predictions,
    references=clean_references,
    lang="en", batch_size=16, device=device_str
)

# B. Para la Referencia (Referencia vs Referencia - Debería dar ~1.0)
print("-> BERTScore: Referencia (Sanity Check)...")
bertscore_results_ref = bertscore_metric.compute(
    predictions=clean_references,
    references=clean_references,
    lang="en", batch_size=16, device=device_str
)

# --- 2. Factualidad (Métrica NLI Invertida) ---
print("\nInicializando modelo NLI...")
nli_model_id = "facebook/bart-large-mnli"
nli_tokenizer = AutoTokenizer.from_pretrained(nli_model_id)
nli_model = AutoModelForSequenceClassification.from_pretrained(nli_model_id).to(device_str)
nli_model.eval()

# Función auxiliar para no repetir código NLI
def calculate_nli(sources_list, summaries_list, desc):
    scores = []
    with torch.no_grad():
        for source_doc, summary in tqdm(zip(sources_list, summaries_list), total=len(summaries_list), desc=desc):
            doc = nlp(summary)
            summary_sentences = [sent.text for sent in doc.sents]

            if not summary_sentences:
                scores.append(0.0)
                continue

            pairs = [[source_doc, sent] for sent in summary_sentences]

            inputs = nli_tokenizer(
                [p[0] for p in pairs], [p[1] for p in pairs],
                max_length=512, truncation=True, padding="longest", return_tensors='pt'
            ).to(device_str)

            outputs = nli_model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1)
            # Index 0 = contradiction
            contradictions = (probs[:, 0] > 0.7).sum().item()
            scores.append(1.0 - (contradictions / len(summary_sentences)))
    return scores

# A. NLI para Modelo
factual_consistency_gen = calculate_nli(clean_sources, clean_predictions, "Factualidad NLI (Modelo)")
# B. NLI para Referencia
factual_consistency_ref = calculate_nli(clean_sources, clean_references, "Factualidad NLI (Referencia)")

# Liberar memoria NLI
del nli_model, nli_tokenizer
torch.cuda.empty_cache()

# --- 4. Factualidad (Métrica QA) ---
print("\nInicializando pipelines de QA...")
q_gen_pipe = pipeline("text2text-generation", model="valhalla/t5-base-qg-hl", device=device)
qa_pipe = pipeline("question-answering", model="distilbert-base-cased-distilled-squad", device=device)

# Función auxiliar para no repetir código QA
def calculate_qa(sources_list, summaries_list, desc):
    scores = []
    for source_doc, summary in tqdm(zip(sources_list, summaries_list), total=len(summaries_list), desc=desc):
        doc = nlp(summary)
        sentences = [sent.text for sent in doc.sents]

        if not sentences:
            scores.append(0.0)
            continue

        try:
            # 1. Generar preguntas
            q_outputs = q_gen_pipe(sentences, max_new_tokens=32, num_beams=2, batch_size=8)
            questions = [item['generated_text'] for item in q_outputs]

            # 2. Responder preguntas usando el TEXTO TÉCNICO como contexto
            qa_inputs = [{'question': q, 'context': source_doc} for q in questions]
            answers = qa_pipe(qa_inputs, batch_size=8)

            valid_answers = sum(1 for ans in answers if ans['score'] > 0.3)
            scores.append(valid_answers / len(sentences))
        except Exception as e:
            # print(f"Err: {e}")
            scores.append(0.0)
    return scores

# A. QA para Modelo
qa_fact_gen = calculate_qa(clean_sources, clean_predictions, "Factualidad QA (Modelo)")
# B. QA para Referencia
qa_fact_ref = calculate_qa(clean_sources, clean_references, "Factualidad QA (Referencia)")

# Liberar memoria QA
del q_gen_pipe, qa_pipe
torch.cuda.empty_cache()

# --- 5. Legibilidad ---
print("\nCalculando métricas de legibilidad...")

def get_readability_metrics(text_list, desc):
    results = {'fkgl': [], 'fre': [], 'gunning_fog': [], 'smog': [], 'coleman_liau': []}
    for text in tqdm(text_list, desc=desc):
        results['fkgl'].append(textstat.flesch_kincaid_grade(text))
        results['fre'].append(textstat.flesch_reading_ease(text))
        results['gunning_fog'].append(textstat.gunning_fog(text))
        results['smog'].append(textstat.smog_index(text))
        results['coleman_liau'].append(textstat.coleman_liau_index(text))
    return results

readability_generated = get_readability_metrics(clean_predictions, "Legibilidad (Modelo)")
readability_reference = get_readability_metrics(clean_references, "Legibilidad (Referencia)")

print("\n¡Cálculos finalizados con éxito!")

# --- 6. Construcción y Guardado de la Tabla Intermedia ---

data = {
    'Métrica': [
        'BERTScore - F1',
        'Factualidad (1 - Contradiction Ratio)',
        'Factualidad (QA Answerability)',
        '---',
        'Flesch-Kincaid Grade',
        'Flesch Reading Ease',
        'Gunning Fog Index',
        'SMOG Index',
        'Coleman-Liau Index',
    ],
    'Meta / Objetivo': ['≥ 0.86', '≥ 0.90', '≥ 0.80', '', '≤ 8.0', '≥ 60', '≤ 12.0', '≤ 8.0', '≤ 8.0'],
    'PLS Reales (Referencia)': [
        f"{np.mean(bertscore_results_ref['f1']):.4f}",   # AHORA CALCULADO
        f"{np.mean(factual_consistency_ref):.4f}",       # AHORA CALCULADO
        f"{np.mean(qa_fact_ref):.4f}",                   # AHORA CALCULADO
        '',
        f"{np.mean(readability_reference['fkgl']):.2f}",
        f"{np.mean(readability_reference['fre']):.2f}",
        f"{np.mean(readability_reference['gunning_fog']):.2f}",
        f"{np.mean(readability_reference['smog']):.2f}",
        f"{np.mean(readability_reference['coleman_liau']):.2f}",
    ],
    'PLS Generados (Modelo)': [
        f"{np.mean(bertscore_results_gen['f1']):.4f}",
        f"{np.mean(factual_consistency_gen):.4f}",
        f"{np.mean(qa_fact_gen):.4f}",
        '',
        f"{np.mean(readability_generated['fkgl']):.2f}",
        f"{np.mean(readability_generated['fre']):.2f}",
        f"{np.mean(readability_generated['gunning_fog']):.2f}",
        f"{np.mean(readability_generated['smog']):.2f}",
        f"{np.mean(readability_generated['coleman_liau']):.2f}",
    ]
}

results_table = pd.DataFrame(data)

# Guardamos con un nombre ESPECÍFICO para recuperarlo en el paso de unificación
PARTIAL_METRICS_PATH = os.path.join(BASE_PROJECT_DIR, 'v8_temp_standard_metrics.csv')
results_table.to_csv(PARTIAL_METRICS_PATH, index=False)

print("\n" + "="*80)
print("--- Tabla Comparativa de Resultados (Métricas Estándar) ---")
print("="*80)
print(results_table.to_string(index=False))
print("="*80)
print(f"Tabla guardada temporalmente en: {PARTIAL_METRICS_PATH}")

In [ ]:
# --- Celda 2.1: Instalar Python 3.10 ---
# Añadimos el repositorio PPA 'deadsnakes' que contiene versiones antiguas de Python
!sudo add-apt-repository -y ppa:deadsnakes/ppa
!sudo apt-get update -y

# Instalamos Python 3.10 y su módulo venv
!sudo apt-get install -qq python3.10 python3.10-venv -y

# Verificamos que se instaló correctamente
!python3.10 --version

In [ ]:
# --- Celda 2.2: Crear y Preparar el Entorno Virtual (VERSIÓN FINAL CORREGIDA) ---

# 1. Creamos un entorno virtual (esto no cambia)
!python3.10 -m venv /content/align_env

# 2. Instalamos PyTorch (esto no cambia)
!/content/align_env/bin/pip install -q torch==1.13.1+cu117 torchvision==0.14.1+cu117 --extra-index-url https://download.pytorch.org/whl/cu117

# 3. ¡EL COMANDO CLAVE Y ÚNICO!
#    Instalamos AlignScore Y TODAS sus dependencias con las versiones correctas en un solo paso.
#    - transformers==4.33.2 (para el import AdamW)
#    - tokenizers==0.13.3 (compatible con transformers 4.33.2)
#    - pytorch-lightning==1.9.5 (la versión que AlignScore espera)
!/content/align_env/bin/pip install -q \
    pandas \
    transformers==4.33.2 \
    tokenizers==0.13.3 \
    pytorch-lightning==1.9.5 \
    "git+https://github.com/yuh-zha/AlignScore.git"

# --- Verificación ---
print("\n--- Verificando el entorno virtual ---")
!/content/align_env/bin/python --version
!/content/align_env/bin/python -c "import torch; print(f'PyTorch version: {torch.__version__}'); import transformers; print(f'Transformers version: {transformers.__version__}'); import alignscore; print('¡ÉXITO! AlignScore y sus dependencias se importaron correctamente.')"

In [ ]:
# --- Celda 2.3: Instalar AlignScore y Descargar el Modelo ---

# Usamos el pip que está DENTRO de nuestro entorno conda para instalar AlignScore
!/usr/local/envs/alignscore_env/bin/pip install -q pandas git+https://github.com/yuh-zha/AlignScore.git

# Clonar el repositorio para tener una estructura de carpetas predecible
!git clone https://github.com/yuh-zha/AlignScore.git

# Descargar el checkpoint del modelo en el directorio clonado
!cd /content/AlignScore/ && wget https://huggingface.co/yzha/AlignScore/resolve/main/AlignScore-base.ckpt

# Verificar que el archivo se descargó
!echo -e "\n--- Verificación de Archivos ---"
!ls -lh /content/AlignScore/

In [ ]:
# 4. ¡NUEVO PASO! Descargamos el modelo de spaCy USANDO el python del entorno virtual.
#    Esto asegura que el modelo sea visible para el script.
!/content/align_env/bin/python -m spacy download en_core_web_sm

# --- Verificación ---
print("\n--- Verificando el entorno virtual ---")
!/content/align_env/bin/python --version
# La verificación ahora también comprueba que spacy puede cargar el modelo.
!/content/align_env/bin/python -c "import torch; print(f'PyTorch version: {torch.__version__}'); import transformers; print(f'Transformers version: {transformers.__version__}'); import alignscore; import spacy; nlp=spacy.load('en_core_web_sm'); print('¡ÉXITO! Todas las dependencias, incluyendo spaCy, se importaron y cargaron correctamente.')"

In [ ]:
%%writefile run_alignscore_dual.py
import os
import pandas as pd
import numpy as np
import torch
from alignscore import AlignScore
import nltk
nltk.download('punkt_tab', quiet=True)

# --- Configuración ---
BASE_PROJECT_DIR = "/content/drive/MyDrive/NLP_Project_MedGemma"
BASE_ALIGNSCORE_DIR = "/content/AlignScore"
ALIGN_SCORE_CKPT_PATH = os.path.join(BASE_ALIGNSCORE_DIR, 'AlignScore-base.ckpt')

# IMPORTANTE: Usar el mismo CSV de entrada que usaste en la celda de métricas estándar
CSV_INPUT_PATH = os.path.join(BASE_PROJECT_DIR, 'v8_debug_audit_summaries.csv')
OUTPUT_SCORE_PATH = os.path.join(BASE_PROJECT_DIR, 'v8_temp_alignscore_results.csv')

print("--- Iniciando cálculo Dual de AlignScore ---")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

try:
    # 1. Cargar Datos
    df = pd.read_csv(CSV_INPUT_PATH)

    # 2. Filtrar columnas requeridas
    required_cols = ['technical_text', 'reference_summary', 'generated_summary']
    for col in required_cols:
        if col not in df.columns:
            if col == 'generated_summary' and 'debug_raw_response' in df.columns:
                df['generated_summary'] = df['debug_raw_response']
            else:
                raise ValueError(f"Falta columna: {col}")

    df_clean = df.dropna(subset=required_cols)
    df_clean = df_clean[
        (df_clean['technical_text'].str.strip() != '') &
        (df_clean['generated_summary'].str.strip() != '')
    ]

    sources = df_clean['technical_text'].tolist()
    refs = df_clean['reference_summary'].tolist()
    gens = df_clean['generated_summary'].tolist()

    # 3. Cargar Modelo
    scorer = AlignScore(model='roberta-large', batch_size=32, device=DEVICE,
                        ckpt_path=ALIGN_SCORE_CKPT_PATH, evaluation_mode='nli_sp')

    # 4. Calcular
    print(f"Calculando AlignScore para {len(sources)} filas...")
    score_gen = np.mean(scorer.score(contexts=sources, claims=gens))
    score_ref = np.mean(scorer.score(contexts=sources, claims=refs))

    print(f"-> Gen: {score_gen:.4f} | Ref: {score_ref:.4f}")

    # 5. Guardar resultados
    pd.DataFrame([
        {'target_type': 'generated', 'score': score_gen},
        {'target_type': 'reference', 'score': score_ref}
    ]).to_csv(OUTPUT_SCORE_PATH, index=False)
    print("Guardado exitoso.")

except Exception as e:
    print(f"ERROR EN ALIGNSCORE: {e}")
    # Guardar ceros de seguridad
    pd.DataFrame([{'target_type': 'generated', 'score': 0}, {'target_type': 'reference', 'score': 0}]).to_csv(OUTPUT_SCORE_PATH, index=False)

Overwriting run_alignscore_dual.py


In [ ]:
import pandas as pd
import os

# Rutas de Archivos (Deben coincidir con los pasos anteriores)
#PARTIAL_METRICS_PATH = os.path.join(BASE_PROJECT_DIR, 'v8_temp_standard_metrics.csv')
#ALIGNSCORE_RES_PATH = os.path.join(BASE_PROJECT_DIR, 'v8_temp_alignscore_results.csv')
#FINAL_REPORT_PATH = os.path.join(BASE_PROJECT_DIR, 'v8_final_metrics_report.csv')

PARTIAL_METRICS_PATH = os.path.join(BASE_PROJECT_DIR, 'v8_temp_standard_metrics.csv')
ALIGNSCORE_RES_PATH = os.path.join(BASE_PROJECT_DIR, 'v8_temp_alignscore_results.csv')
FINAL_REPORT_PATH = os.path.join(BASE_PROJECT_DIR, 'v8_final_metrics_report.csv')

# --- 1. Ejecutar AlignScore ---
print("="*80)
print("Calculando AlignScore en entorno aislado...")
!/content/align_env/bin/python run_alignscore_dual.py
print("="*80)

# --- 2. Carga y Fusión Automática ---
try:
    # A. Cargar Tabla de Métricas Estándar
    df_main = pd.read_csv(PARTIAL_METRICS_PATH)

    # B. Cargar Resultado de AlignScore
    df_align = pd.read_csv(ALIGNSCORE_RES_PATH)
    as_gen = df_align.loc[df_align['target_type'] == 'generated', 'score'].values[0]
    as_ref = df_align.loc[df_align['target_type'] == 'reference', 'score'].values[0]

    # C. Crear la fila de AlignScore con el mismo formato que df_main
    # Obtenemos los nombres de columnas dinámicamente para evitar errores
    cols = df_main.columns.tolist()

    new_row = pd.DataFrame([{
        cols[0]: 'AlignScore (NLI_SP)',   # Columna 'Métrica'
        cols[1]: '≥ 0.80',                # Columna 'Meta / Objetivo'
        cols[2]: f"{as_ref:.4f}",         # Columna 'PLS Reales (Referencia)'
        cols[3]: f"{as_gen:.4f}"          # Columna 'PLS Generados (Modelo)'
    }])

    # D. Insertar la fila en la posición deseada (Después de BERTScore, índice 1)
    # Dividimos el dataframe y ponemos el sandwich en el medio
    df_final = pd.concat([df_main.iloc[:1], new_row, df_main.iloc[1:]]).reset_index(drop=True)

    # --- 3. Guardar e Imprimir ---
    df_final.to_csv(FINAL_REPORT_PATH, index=False)

    print("\n" + "="*105)
    print("--- Tabla Comparativa Final de Resultados (Unificada Automáticamente) ---")
    print("="*105)

    # Formateo dinámico para impresión limpia
    # Calculamos ancho máximo para alineación visual
    col_widths = [max(df_final[c].astype(str).map(len).max(), len(c)) + 2 for c in df_final.columns]
    header = "".join(f"{c:<{w}}" for c, w in zip(df_final.columns, col_widths))

    print(header)
    print("-" * len(header))

    for _, row in df_final.iterrows():
        if row[cols[0]] == '---':
            print("-" * len(header))
        else:
            print("".join(f"{str(row[c]):<{w}}" for c, w in zip(df_final.columns, col_widths)))

    print("="*105)
    print(f"Reporte final guardado en: {FINAL_REPORT_PATH}")

except FileNotFoundError as e:
    print(f"\nERROR CRÍTICO: Falta uno de los archivos intermedios. {e}")
except Exception as e:
    print(f"\nERROR INESPERADO: {e}")

In [ ]:
# =======================================================================
# FASE 5: EVALUACIÓN CUALITATIVA COMPARATIVA (Gemini vs. Llama) - CORREGIDO
# =======================================================================
#
# Este script modificado carga los resúmenes generados por Qwen y Gemma
# del archivo [pls_generated_qwen_gemma.csv].
#
# Utiliza dos modelos "jueces" (Gemini 1.5 Flash y Llama 3.1 8B Instruct)
# para realizar una evaluación cualitativa de AMBOS resúmenes generados.
#
# CORRECCIONES v2:
# 1.  Se cambió 'gemini-1.5-flash' por 'gemini-1.5-flash-lite' para
#     solucionar el error 404 de la API de Google.
# 2.  Se cambió `llama_client.chat.completions(...)` por
#     `llama_client.chat.completions.create(...)` para solucionar
#     el error "Not Callable" de la API de Hugging Face.
#
# =======================================================================

# --- Paso 1: Instalación y Configuración ---

# Instalar bibliotecas de cliente
!pip install -q google-generativeai huggingface_hub

print("Instalando e importando librerías...")
import google.generativeai as genai
from huggingface_hub import HfApi, InferenceClient, login
from google.colab import userdata
import pandas as pd
import os
import json
import re  # Importar regex para la extracción de JSON de Llama
import time
from tqdm.auto import tqdm

# Aplicar tqdm a las operaciones de pandas
tqdm.pandas(desc="Procesando filas")

Instalando e importando librerías...


In [ ]:
# Celda 4: Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 6. Evaluación cualitativa y profiling
Define claves y rutas para Gemini, inicializa funciones de evaluación, instala y ejecuta ydata-profiling para revisar resultados cualitativos.

In [ ]:
# --- Configuración de API Key de Google (Gemini) ---
print("Configurando la API Key de Gemini...")
try:
    API_KEY_GOOGLE = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=API_KEY_GOOGLE)
    print("API Key de Google (Gemini) configurada exitosamente.")
except Exception as e:
    print(f"Error al configurar la API Key de Google: {e}")
    print("Por favor, asegúrate de que 'GOOGLE_API_KEY' esté guardada en los 'Secrets' de Colab (icono de la llave).")
    raise

# --- Configuración de API Key de Hugging Face ---
print("Configurando la API Key de HuggingFace...")
try:
    API_KEY_HF = userdata.get('HUGGING_API_KEY')
    login(token=API_KEY_HF)
    print("API Key de HuggingFace configurada exitosamente.")
except Exception as e:
    print(f"Error al configurar la API Key de Hugging Face: {e}")
    print("Por favor, asegúrate de que 'HUGGING_API_KEY' esté guardada en los 'Secrets' de Colab (icono de la llave).")
    raise

Configurando la API Key de Gemini...
API Key de Google (Gemini) configurada exitosamente.
Configurando la API Key de HuggingFace...
API Key de HuggingFace configurada exitosamente.


In [ ]:
# --- Paso 2: Definiciones de Rutas y Carga de Datos ---

# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# IMPORTANTE: Asegúrate de que la variable BASE_PROJECT_DIR esté definida
# en una celda anterior, como en tu script original.
# Ejemplo:
# BASE_PROJECT_DIR = '/content/drive/MyDrive/TuProyecto'
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
BASE_PROJECT_DIR = "/content/drive/MyDrive/NLP_Project_MedGemma"

if 'BASE_PROJECT_DIR' not in locals():
    print("ADVERTENCIA: La variable 'BASE_PROJECT_DIR' no está definida.")
    print("Definiéndola como '/content/' por defecto. Edita esto si es necesario.")
    BASE_PROJECT_DIR = '/content/'

# **MODIFICADO**: Ruta al nuevo archivo CSV de entrada
csv_path = os.path.join(BASE_PROJECT_DIR, 'v7_debug_audit_summaries.csv')

# **MODIFICADO**: Rutas para los DOS nuevos archivos CSV de salida
output_csv_gemini_path = os.path.join(BASE_PROJECT_DIR, 'v7_gemini_qualitative_eval.csv')

print(f"Cargando datos desde: {csv_path}")
try:
    df = pd.read_csv(csv_path)
except FileNotFoundError:
    print(f"ERROR: No se encontró el archivo en la ruta: {csv_path}")
    print(f"Asegúrate de que el archivo {csv_path} esté en el lugar correcto.")
    raise

# **MODIFICADO**: Limpieza básica para las nuevas columnas
columns_to_check = ['technical_text', 'reference_summary', 'generated_summary', 'debug_raw_response']
df.dropna(subset=columns_to_check, inplace=True)
df = df[df['generated_summary'].str.strip() != '']
df = df[df['debug_raw_response'].str.strip() != '']

print(f"Cargados {len(df)} ejemplos para la evaluación cualitativa.")

Cargando datos desde: /content/drive/MyDrive/NLP_Project_MedGemma/v7_debug_audit_summaries.csv
Cargados 100 ejemplos para la evaluación cualitativa.


In [ ]:
# --- Paso 3: Diseño del Prompt de Evaluación y Plantilla JSON ---

PROMPT_DE_EVALUACION = """
Eres un evaluador experto en lingüística y comunicación de la ciencia.
Tu tarea es evaluar un "Resumen Generado" (Plain Language Summary - PLS).

**Contexto del Proyecto:**
El objetivo es crear un resumen con un nivel de lectura de 8.º grado (meta: Flesch-Kincaid Grade <= 8.0) que sea factualmente preciso y no omita información crítica.

Tu misión es analizar por qué el resumen generado podría fallar o tener éxito.

**Textos para Evaluación:**

1.  **Texto Técnico (Fuente):**
    ```
    {technical_text}
    ```

2.  **Resumen de Referencia (PLS Humano):**
    ```
    {reference_summary}
    ```

3.  **Resumen Generado (PLS del Modelo a Evaluar):**
    ```
    {summary_to_evaluate}
    ```

**Tareas de Evaluación:**
Basado en los textos, evalúa el "Resumen Generado" (PLS del Modelo a Evaluar) y responde ÚNICAMENTE con un objeto JSON válido que siga esta estructura:

{{
  "legibilidad_cumple_meta_8vo_grado": false,
  "analisis_legibilidad": "El texto sigue usando jerga técnica como 'XYZ' y la estructura de las frases es compleja. Nivel estimado: Grado 11.",
  "es_factualmente_consistente": true,
  "factual_completitud_cumple_meta": false,
  "analisis_factual": "El resumen es correcto, pero omite la conclusión principal del estudio (el resultado X). Es demasiado vago.",
  "tiene_errores_gramaticales": false,
  "tiene_errores_ortograficos": false,
  "es_coherente": true,
  "tiene_repeticiones": false,
  "tiene_frases_incompletas": false,
  "resumen_evaluacion_general": "El resumen generado simplifica el lenguaje pero pierde el punto central del texto técnico, haciéndolo inútil para entender los resultados clave."
}}

**Instrucciones del JSON:**
* Usa `true` o `false` para los campos booleanos.
* Proporciona tu análisis experto en las cadenas de texto (`analisis_legibilidad`, `analisis_factual`, `resumen_evaluacion_general`).
* `analisis_factual`: Enfócate en si es vago u omite información clave.
* `analisis_legibilidad`: Enfócate en por qué fallaría la meta de 8.º grado.
* **Responde SÓLO con el objeto JSON, nada más.**
"""

# Estructura JSON esperada para manejo de errores
EXPECTED_JSON_STRUCTURE = {
  "legibilidad_cumple_meta_8vo_grado": None,
  "analisis_legibilidad": "ERROR: No se pudo generar la evaluación.",
  "es_factualmente_consistente": None,
  "factual_completitud_cumple_meta": None,
  "analisis_factual": "ERROR: No se pudo generar la evaluación.",
  "tiene_errores_gramaticales": None,
  "tiene_errores_ortograficos": None,
  "es_coherente": None,
  "tiene_repeticiones": None,
  "tiene_frases_incompletas": None,
  "resumen_evaluacion_general": "ERROR: La llamada a la API falló."
}

In [ ]:
# --- Paso 4: Inicialización de Modelos y Funciones de Evaluación ---

# --- Modelo 1: Gemini ---
print("Inicializando modelo Gemini...")
# **CORRECCIÓN 1**: Se usa 'gemini-1.5-flash-latest' para evitar el error 404
gemini_model = genai.GenerativeModel('gemini-2.5-flash-lite')

# --- Funciones de Utilidad ---

def get_error_response(error_message):
    """Devuelve un diccionario de error con la estructura JSON esperada."""
    error_json = EXPECTED_JSON_STRUCTURE.copy()
    error_json["resumen_evaluacion_general"] = f"ERROR: {error_message}"
    return error_json

def extract_json_from_text(text):
    """
    Extrae el primer objeto JSON válido de una cadena de texto,
    incluso si está rodeado de texto o markdown.
    """
    # Buscar el primer '{' y el último '}' para capturar el JSON
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if not match:
        # Si no encuentra JSON, buscar la palabra "ERROR" para propagar fallos
        if "error" in text.lower():
             raise ValueError(f"Respuesta de Llama contenía un error: {text}")
        raise ValueError("No se encontró un objeto JSON en la respuesta de Llama.")

    json_string = match.group(0)
    return json.loads(json_string)

# --- Funciones de Evaluación Principales ---

def evaluate_with_gemini(row, summary_column_name):
    """
    Llama a la API de Gemini para evaluar un resumen específico
    (dado por summary_column_name).
    """
    try:
        # 1. Construir el prompt
        prompt = PROMPT_DE_EVALUACION.format(
            technical_text=row['technical_text'],
            reference_summary=row['reference_summary'],
            summary_to_evaluate=row[summary_column_name] # Usa la columna pasada
        )

        # 2. Configurar el modelo para que devuelva JSON
        generation_config = genai.types.GenerationConfig(
            response_mime_type="application/json"
        )

        # 3. Llamar a la API
        response = gemini_model.generate_content(prompt, generation_config=generation_config)

        # 4. Parsear la respuesta JSON
        result_json = json.loads(response.text)

        # 5. Validar claves
        for key in EXPECTED_JSON_STRUCTURE.keys():
            if key not in result_json:
                raise ValueError(f"La respuesta JSON de Gemini no contenía la clave: '{key}'")

        return result_json

    except Exception as e:
        print(f"Error procesando (Gemini) fila {row.name}, col {summary_column_name}: {str(e)}")
        return get_error_response(f"Gemini API Error: {str(e)}")

Inicializando modelo Gemini...


In [ ]:
import time
import random
import pandas as pd
from tqdm import tqdm

# --- Configuración de Robustez ---
MAX_RETRIES = 3          # Cuántas veces reintentar si falla
INITIAL_BACKOFF = 2      # Segundos iniciales de espera tras error
GEMINI_API_DELAY = 4.1   # Tu delay estándar para Rate Limits (RPM)
SAVE_EVERY_N_ROWS = 5    # Guardar respaldo cada 5 filas (Checkpoint)
OUTPUT_FILENAME = os.path.join(BASE_PROJECT_DIR, 'partial_v7_gemini_qualitative_eval.csv')

# Claves que SIEMPRE deben existir en la respuesta JSON (basado en tu error)
REQUIRED_KEYS = ['tiene_errores_ortograficos']
# (Agrega aquí otras claves críticas que esperas, ej: 'puntaje', 'razonamiento')

# --- Función Wrapper para Reintentos ---
def safe_evaluate_with_retry(row, col_name, retries=MAX_RETRIES):
    """
    Intenta llamar a evaluate_with_gemini con reintentos y validación de JSON.
    """
    attempt = 0
    last_exception = None

    while attempt < retries:
        try:
            # 1. Intentar llamada
            response_json = evaluate_with_gemini(row, col_name)

            # 2. Validar si el resultado es un diccionario válido
            if not isinstance(response_json, dict):
                raise ValueError(f"La respuesta no es un dict JSON válido: {type(response_json)}")

            # 3. Validar que contenga las claves críticas (tu error específico)
            missing_keys = [k for k in REQUIRED_KEYS if k not in response_json]
            if missing_keys:
                raise KeyError(f"Faltan claves en JSON: {missing_keys}")

            # Si todo sale bien, retornamos el JSON limpio
            return response_json, None # (resultado, error)

        except Exception as e:
            attempt += 1
            last_exception = e

            # Calcular tiempo de espera con un poco de 'jitter' (variación aleatoria)
            # para evitar colisiones si corrieras hilos paralelos.
            sleep_time = (INITIAL_BACKOFF * (2 ** (attempt - 1))) + random.uniform(0, 1)

            print(f" > Error en intento {attempt}/{retries} para {col_name}: {e}")
            print(f" > Reintentando en {sleep_time:.2f}s...")
            time.sleep(sleep_time)

    # Si agotamos los intentos, devolvemos un diccionario de error y la excepción
    error_result = {k: "ERROR_API" for k in REQUIRED_KEYS}
    error_result['error_detail'] = str(last_exception)
    return error_result, last_exception


# --- Paso 5: Ejecución del Bucle (Mejorado) ---

print("\n" + "="*80)
print(f"Iniciando evaluación ROBUSTA de {len(df)} filas con Gemini...")
print(f"Retardo base: {GEMINI_API_DELAY}s | Reintentos máx: {MAX_RETRIES}")
print("="*80)

gemini_results_data = []

# Iterar sobre el DataFrame
for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Evaluando filas"):

    base_data = row.to_dict()
    row_errors = False # Bandera para saber si hubo error grave en esta fila

    # --- A: Evaluar 'generated_summary' ---
    # Usamos la función wrapper en lugar de llamar directo
    generated_summary_json, err_gen_summ = safe_evaluate_with_retry(row, 'generated_summary')

    # Construimos el objeto incluso si hubo error (para no perder la fila)
    gemini_row_generated_summary = {
        **base_data,
        **generated_summary_json,
        'evaluated_summary_model': 'generated_summary',
        'api_status': 'failed' if err_gen_summ else 'success'
    }
    gemini_results_data.append(gemini_row_generated_summary)

    # Respetar límite de velocidad API (solo si fue exitoso, si falló ya esperamos bastante)
    if not err_gen_summ:
        time.sleep(GEMINI_API_DELAY)

    # --- B: Evaluar 'pls_qwen' ---
    debug_raw_response_json, err_deb_rawres = safe_evaluate_with_retry(row, 'debug_raw_response')

    gemini_row_deb_rawres = {
        **base_data,
        **debug_raw_response_json,
        'evaluated_summary_model': 'debug_raw_response',
        'api_status': 'failed' if err_deb_rawres else 'success'
    }
    gemini_results_data.append(gemini_row_deb_rawres)

    if not err_deb_rawres:
        time.sleep(GEMINI_API_DELAY)

    # --- C: Guardado Intermedio (Checkpoint) ---
    # Esto salva la vida si se va la luz o crashea el kernel
    if (index + 1) % SAVE_EVERY_N_ROWS == 0:
        temp_df = pd.DataFrame(gemini_results_data)
        temp_df.to_csv(OUTPUT_FILENAME, index=False)
        # Opcional: print(f"Checkpoint guardado en fila {index}")

# --- Guardado Final ---
print("\nEvaluación completada. Consolidando resultados finales...")
final_df = pd.DataFrame(gemini_results_data)

# Verificar si hubo fallos
failed_count = final_df[final_df['api_status'] == 'failed'].shape[0]
if failed_count > 0:
    print(f"⚠️ ATENCIÓN: Hubo {failed_count} evaluaciones fallidas que agotaron los reintentos.")
    print("Revisa la columna 'error_detail' en el CSV resultante.")
else:
    print("✅ Éxito total: Todas las filas procesadas sin errores persistentes.")

# Guardar
final_df.to_csv(output_csv_gemini_path, index=False)
print(f"Resultados guardados en: {output_csv_gemini_path}")


Iniciando evaluación ROBUSTA de 100 filas con Gemini...
Retardo base: 4.1s | Reintentos máx: 3


Evaluando filas:  91%|█████████ | 91/100 [20:23<01:55, 12.82s/it]

Error procesando (Gemini) fila 91, col generated_summary: La respuesta JSON de Gemini no contenía la clave: 'tiene_errores_ortograficos'


Evaluando filas: 100%|██████████| 100/100 [22:17<00:00, 13.38s/it]



Evaluación completada. Consolidando resultados finales...
✅ Éxito total: Todas las filas procesadas sin errores persistentes.
Resultados guardados en: /content/drive/MyDrive/NLP_Project_MedGemma/v7_gemini_qualitative_eval.csv


In [ ]:
!pip install -q ydata-profiling

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 399.3/399.3 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.5/296.5 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 1.0 MB/s eta 0:00:00


In [ ]:
# Conectarse a Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import pandas as pd
import os
BASE_PROJECT_DIR = "/content/drive/MyDrive/NLP_Project_MedGemma"
# Construct the full path to the CSV file
csv_path = os.path.join(BASE_PROJECT_DIR, 'v8_gemini_qualitative_eval.csv')

print(f"Loading data from: {csv_path}")

# Load the CSV file into a pandas DataFrame
df_eval = pd.read_csv(csv_path)
# Filtering by evaluated_summary_model = 'pls_gemma'
df_eval = df_eval[df_eval['evaluated_summary_model'] == 'generated_summary']
# Droping columns technical_text, reference_summary, pls_gemma, and pls_qwen
df_eval = df_eval.drop(columns=['technical_text', 'reference_summary', 'generated_summary', 'debug_full_text', 'debug_raw_response', 'evaluated_summary_model', 'api_status'])

# Display the first few rows of the DataFrame
print("\nFirst 5 rows of the loaded DataFrame:")
display(df_eval.head())

# Print the shape of the DataFrame
print("\nShape of the loaded DataFrame:")
print(df_eval.shape)

Loading data from: /content/drive/MyDrive/NLP_Project_MedGemma/v8_gemini_qualitative_eval.csv

First 5 rows of the loaded DataFrame:


,legibilidad_cumple_meta_8vo_grado,analisis_legibilidad,es_factualmente_consistente,factual_completitud_cumple_meta,analisis_factual,tiene_errores_gramaticales,tiene_errores_ortograficos,es_coherente,tiene_repeticiones,tiene_frases_incompletas,resumen_evaluacion_general
0,False,El resumen utiliza lenguaje técnico y concepto...,True,False,"Aunque el resumen es factualmente consistente,...",False,False,True,False,False,El resumen generado logra simplificar parte de...
2,False,El resumen utiliza lenguaje relativamente acce...,True,False,El resumen generado omite la conclusión princi...,False,False,True,True,False,El resumen intenta simplificar el lenguaje per...
4,False,El resumen generado utiliza términos técnicos ...,True,False,El resumen omite la matizada conclusión de que...,False,False,True,False,False,El resumen generado es factualmente consistent...
6,False,El resumen utiliza un lenguaje relativamente s...,True,False,El resumen es factualmente correcto en lo que ...,False,False,True,False,False,El resumen generado intenta simplificar el len...
8,False,El resumen usa frases relativamente cortas y u...,True,False,El resumen omite la comparación directa de la ...,False,False,True,False,False,El resumen genera una comprensión básica de qu...



Shape of the loaded DataFrame:
(100, 11)


In [ ]:
from ydata_profiling import ProfileReport

# Create a ProfileReport object
profile = ProfileReport(df_eval, title="Qualitative Evaluation Data Profiling Report")

In [ ]:
# Save the report as an HTML file
report_path = os.path.join(BASE_PROJECT_DIR, "v8_qualitative_gemini_gemma_eval_report.html")
profile.to_file(report_path)

print(f"Profiling report saved to: {report_path}")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 11/11 [00:00<00:00, 92.14it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Profiling report saved to: /content/drive/MyDrive/NLP_Project_MedGemma/v8_qualitative_gemini_gemma_eval_report.html


In [ ]:
from IPython.display import HTML

# Construct the full path to the saved HTML report file
report_path = os.path.join(BASE_PROJECT_DIR, "v8_qualitative_gemini_gemma_eval_report.html")

# Open and read the content of the HTML file
with open(report_path, 'r', encoding='utf-8') as f:
    html_content = f.read()

# Display the HTML content directly in the notebook
display(HTML(html_content))

## 7. Generación con modelos fundacionales
Monta Drive, define prompts de generación, ejecuta bucles de inferencia y guarda resultados con dependencias mínimas.

In [ ]:
# =======================================================================
# FASE 6: GENERACIÓN DE PLS CON MODELOS FUNDACIONALES (CON REINTENTOS)
# =======================================================================
#
# Este script carga el texto técnico de origen y utiliza los modelos
# fundacionales (Gemini 1.5 Flash y Llama 3.1 8B Instruct) para
# generar nuevos "Plain Language Summaries" (PLS).
#
# =======================================================================
# NOVEDAD (v2):
# Se han modificado las funciones `generate_pls_with_gemini` y
# `generate_pls_with_llama` para incluir un bucle de reintentos
# automático (hasta 3 intentos) con espera (backoff) en caso de
# cualquier error de API (timeout, 500, etc.) o error de validación.
# Esto asegura una mayor robustez y reduce la necesidad de
# intervenciones manuales.
# =======================================================================

# --- Paso 1: Instalación y Configuración ---

# Instalar bibliotecas de cliente
!pip install -q google-generativeai huggingface_hub

print("Instalando e importando librerías...")
import google.generativeai as genai
from huggingface_hub import HfApi, InferenceClient, login
from google.colab import userdata
import pandas as pd
import os
import json
import time
from tqdm.auto import tqdm

# Aplicar tqdm a las operaciones de pandas
tqdm.pandas(desc="Procesando filas")

Instalando e importando librerías...


In [ ]:
# --- Configuración de API Key de Google (Gemini) ---
print("Configurando la API Key de Gemini...")
try:
    API_KEY_GOOGLE = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=API_KEY_GOOGLE)
    print("API Key de Google (Gemini) configurada exitosamente.")
except Exception as e:
    print(f"Error al configurar la API Key de Google: {e}")
    print("Asegúrate de que 'GOOGLE_API_KEY' esté guardada en los 'Secrets'.")
    raise

# --- Configuración de API Key de Hugging Face (Llama) ---
print("Configurando la API Key de HuggingFace...")
try:
    API_KEY_HF = userdata.get('HUGGING_API_KEY')
    login(token=API_KEY_HF)
    print("API Key de HuggingFace (Llama) configurada exitosamente.")
except Exception as e:
    print(f"Error al configurar la API Key de Hugging Face: {e}")
    print("Asegúrate de que 'HUGGING_API_KEY' esté guardada en los 'Secrets'.")
    raise

Configurando la API Key de Gemini...
API Key de Google (Gemini) configurada exitosamente.
Configurando la API Key de HuggingFace...
API Key de HuggingFace (Llama) configurada exitosamente.


In [ ]:
# Celda 4: Montar Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# --- Paso 2: Definiciones de Rutas y Carga de Datos ---

# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# IMPORTANTE: Asegúrate de que la variable BASE_PROJECT_DIR esté definida
# en una celda anterior.
# Ejemplo:
# BASE_PROJECT_DIR = '/content/drive/MyDrive/TuProyecto'
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

BASE_PROJECT_DIR = "/content/drive/MyDrive/NLP_Project_MedGemma"

if 'BASE_PROJECT_DIR' not in locals():
    print("ADVERTENCIA: La variable 'BASE_PROJECT_DIR' no está definida.")
    print("Definiéndola como '/content/' por defecto. Edita esto si es necesario.")
    BASE_PROJECT_DIR = '/content/'

# Ruta al archivo CSV de entrada (el mismo de la Fase 5)
csv_path_input = os.path.join(BASE_PROJECT_DIR, 'v8_debug_audit_summaries.csv')

# Ruta para el NUEVO archivo CSV de salida
csv_path_output = os.path.join(BASE_PROJECT_DIR, 'v8_foundation_model_pls_results.csv')

print(f"Cargando datos desde: {csv_path_input}")
try:
    df = pd.read_csv(csv_path_input)
except FileNotFoundError:
    print(f"ERROR: No se encontró el archivo en la ruta: {csv_path_input}")
    raise

# Limpieza: Solo necesitamos textos técnicos que no estén vacíos
df.dropna(subset=['technical_text', 'reference_summary'], inplace=True)
df = df[df['technical_text'].str.strip() != '']

# Tomamos solo las columnas que nos interesan para el nuevo archivo
df = df[['technical_text', 'reference_summary', 'generated_summary']].drop_duplicates().reset_index(drop=True)

print(f"Cargados {len(df)} textos técnicos únicos para generar PLS.")

Cargando datos desde: /content/drive/MyDrive/NLP_Project_MedGemma/v8_debug_audit_summaries.csv
Cargados 100 textos técnicos únicos para generar PLS.


In [ ]:
# --- Paso 3: Definición del Prompt de Generación ---

SYSTEM_PROMPT = (
        "You are a Health Literacy Expert."
        "Your expertise is in rewriting complex, technical medical texts into clear, simple, and accurate language for a general audience, following established health communication guidelines."
)

INSTRUCTION_TEMPLATE = """**[PRIMARY GOAL]**
        Your main purpose is to rewrite the provided medical text into a Plain Language Summary (PLS).
        This summary must be easy to understand for someone with an 8th-grade reading comprehension level typical of a general middle-school student,
        consistent with plain-language standards used by CDC and NIH, meaning it should use simple vocabulary, short sentences,
        and concepts that can be understood by someone with basic middle-school literacy,
        while remaining completely faithful to the source's essential information, for intance, conclusions.

        **[TASK INSTRUCTION]**
        Rewrite the following technical medical text into a Plain Language Summary (PLS).
        The output **MUST BE ONLY** the plain language summary without special symbols and stop tokens.

        **--- STRICT OUTPUT RULES ---**

        1.  **ACCURACY AND COMPLETENESS:**
        *   The summary MUST retain all key findings, main outcomes, important safety information, conclusions,and any significant numerical results from the original text.
        *   Do NOT add any information, opinions, or recommendations that are not present in the source document. The summary must be based ONLY on the provided text.

        2.  **CLARITY AND READABILITY:**
        *   Write the summary at an **8th-grade reading comprehension level typical of a general middle-school student**.
        *   Use short, clear, and natural-sounding sentences.
        *   Use the active voice whenever possible (e.g., "Scientists tested the drug" instead of "The drug was tested by scientists").
        *   Avoid long, complex words when a simpler alternative exists.

        3.  **TERMINOLOGY (JARGON):**
        *   Avoid medical jargon.
        *   If a technical term is absolutely essential and cannot be replaced, you MUST explain it simply in parentheses the first time it appears. (Example: "The trial used immunotherapy (a treatment that helps the body's immune system fight cancer).")

        4.  **FORMATTING AND LANGUAGE:**
        *   The output **MUST BE ONLY** the plain language summary without special symbols and stop tokens, **only the summary**.
        *   The output must be written **ONLY in English**.
        *   Structure the summary as a set of concise paragraphs. Use as many sentences as needed to include all essential information, up to a maximum length of about 500 words, however, **make sure ALL sentences are complete, which means ALL ideas are finished.**
        *   Do NOT include headings, bullet points, lists, citations, or URLs.

        **--- SOURCE TECHNICAL TEXT ---**
        <document>
        {technical_text}
        </document>

        **--- PLAIN LANGUAGE SUMMARY (PLS) ---**
      """

print("Prompts de generación definidos.")

Prompts de generación definidos.


In [ ]:
# --- Paso 4: Inicialización de Modelos y Funciones de Generación ---

# --- Modelo 1: Gemini ---
print("Inicializando modelo Gemini...")
try:
    gemini_model = genai.GenerativeModel(
        'gemini-2.5-flash-lite',
        system_instruction=SYSTEM_PROMPT
    )
    print("Modelo Gemini (gemini-2.5-flash-lite) inicializado con System Prompt.")
except Exception as e:
    print(f"Error al inicializar Gemini: {e}")
    raise

# --- Modelo 2: Llama 3.1 ---
print("Inicializando cliente de inferencia para Llama...")
LLAMA_MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
llama_client = InferenceClient(model=LLAMA_MODEL_ID)

# --- Funciones de Generación (CON REINTENTOS) ---

def generate_pls_with_gemini(technical_text, max_retries=3, base_delay=3):
    """
    Genera un PLS usando el modelo de Gemini, con reintentos.
    """
    for attempt in range(max_retries):
        try:
            # Formateamos la instrucción del usuario
            prompt_content = INSTRUCTION_TEMPLATE.format(technical_text=technical_text)

            # Generamos el contenido (el system_prompt ya está configurado)
            response = gemini_model.generate_content(prompt_content)

            # Validación: si la respuesta está vacía, forzar reintento
            response_text = response.text.strip()
            if not response_text:
                raise ValueError("Respuesta de Gemini vacía.")

            return response_text # Éxito, salir de la función

        except Exception as e:
            print(f"Error en Gemini (Intento {attempt + 1}/{max_retries}): {str(e)}")
            if attempt < max_retries - 1:
                # Espera exponencial: 3s, 6s
                wait_time = base_delay * (attempt + 1)
                print(f"Reintentando en {wait_time} segundos...")
                time.sleep(wait_time)
            else:
                print("Fallaron todos los reintentos para Gemini.")
                return f"ERROR_GEMINI_FINAL: {str(e)}"

def generate_pls_with_llama(technical_text, max_retries=3, base_delay=3):
    """
    Genera un PLS usando Llama 3.1, con reintentos.
    """
    for attempt in range(max_retries):
        try:
            # Formateamos la instrucción del usuario
            prompt_user = INSTRUCTION_TEMPLATE.format(technical_text=technical_text)

            # Creamos la lista de mensajes (Sistema + Usuario)
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt_user}
            ]

            # Llamamos a la API de Inferencia
            response = llama_client.chat.completions.create(
                messages=messages,
                model=LLAMA_MODEL_ID,
                max_tokens=1024,
                temperature=0.2,
            )

            response_text = response.choices[0].message.content.strip()

            # Validación: si la respuesta está vacía, forzar reintento
            if not response_text:
                 raise ValueError("Respuesta de Llama vacía.")

            return response_text # Éxito, salir de la función

        except Exception as e:
            print(f"Error en Llama (Intento {attempt + 1}/{max_retries}): {str(e)}")
            if attempt < max_retries - 1:
                # Espera exponencial: 3s, 6s
                wait_time = base_delay * (attempt + 1)
                print(f"Reintentando en {wait_time} segundos...")
                time.sleep(wait_time)
            else:
                print("Fallaron todos los reintentos para Llama.")
                return f"ERROR_LLAMA_FINAL: {str(e)}"

Inicializando modelo Gemini...
Modelo Gemini (gemini-2.5-flash-lite) inicializado con System Prompt.
Inicializando cliente de inferencia para Llama...


In [ ]:
# --- Paso 5: Ejecución del Bucle de Generación ---

print("\n" + "="*80)
print(f"Iniciando la generación de PLS para {len(df)} textos...")

# 1. Definir límites de frecuencia (estos son delays *entre* filas exitosas)
GEMINI_API_DELAY = 4.1 # 15 RPM
LLAMA_API_DELAY = 1.0  # Delay genérico para la API de HF

print(f"Retardo (entre filas) para Gemini: {GEMINI_API_DELAY}s")
print(f"Retardo (entre filas) para Llama: {LLAMA_API_DELAY}s")

tiempo_por_fila = GEMINI_API_DELAY + LLAMA_API_DELAY
tiempo_estimado_min = (len(df) * tiempo_por_fila) / 60
print(f"Tiempo total estimado (mínimo): ~{tiempo_estimado_min:.1f} minutos.")
print("(El tiempo real puede ser mayor si ocurren reintentos)")
print("="*80)

# 2. Crear listas para guardar los nuevos resúmenes
gemini_pls_list = []
llama_pls_list = []

# 3. Iterar sobre el DataFrame
for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Generando PLS"):

    current_technical_text = row['technical_text']
    print(f"\n--- Procesando Fila {index} ---")

    # --- A: Generar con Gemini (con reintentos internos) ---
    print(f"Fila {index}: Generando con Gemini...")
    pls_gemini = generate_pls_with_gemini(current_technical_text)
    gemini_pls_list.append(pls_gemini)

    # Esperar *después* de que la función termine (éxito o fallo final)
    time.sleep(GEMINI_API_DELAY)

    # --- B: Generar con Llama (con reintentos internos) ---
    print(f"Fila {index}: Generando con Llama...")
    pls_llama = generate_pls_with_llama(current_technical_text)
    llama_pls_list.append(pls_llama)

    # Esperar *después* de que la función termine (éxito o fallo final)
    time.sleep(LLAMA_API_DELAY)

print("\nGeneración completada. Consolidando resultados...")


Iniciando la generación de PLS para 100 textos...
Retardo (entre filas) para Gemini: 4.1s
Retardo (entre filas) para Llama: 1.0s
Tiempo total estimado (mínimo): ~8.5 minutos.
(El tiempo real puede ser mayor si ocurren reintentos)


Generando PLS:   0%|          | 0/100 [00:00<?, ?it/s]


--- Procesando Fila 0 ---
Fila 0: Generando con Gemini...
Fila 0: Generando con Llama...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



--- Procesando Fila 1 ---
Fila 1: Generando con Gemini...
Fila 1: Generando con Llama...

--- Procesando Fila 2 ---
Fila 2: Generando con Gemini...
Fila 2: Generando con Llama...

--- Procesando Fila 3 ---
Fila 3: Generando con Gemini...
Fila 3: Generando con Llama...

--- Procesando Fila 4 ---
Fila 4: Generando con Gemini...
Fila 4: Generando con Llama...

--- Procesando Fila 5 ---
Fila 5: Generando con Gemini...
Fila 5: Generando con Llama...

--- Procesando Fila 6 ---
Fila 6: Generando con Gemini...
Fila 6: Generando con Llama...

--- Procesando Fila 7 ---
Fila 7: Generando con Gemini...
Fila 7: Generando con Llama...

--- Procesando Fila 8 ---
Fila 8: Generando con Gemini...
Fila 8: Generando con Llama...

--- Procesando Fila 9 ---
Fila 9: Generando con Gemini...
Fila 9: Generando con Llama...

--- Procesando Fila 10 ---
Fila 10: Generando con Gemini...
Fila 10: Generando con Llama...

--- Procesando Fila 11 ---
Fila 11: Generando con Gemini...
Fila 11: Generando con Llama...

---

In [ ]:
# --- Paso 6: Consolidación y Guardado de Resultados ---
try:
    # 1. Crear un DataFrame con los resultados
    df_results = pd.DataFrame({
        'gemini_pls': gemini_pls_list,
        'llama_pls': llama_pls_list
    }, index=df.index)

    # 2. Unir con el DataFrame original (que tiene technical_text y reference_summary)
    df_final = pd.concat([df, df_results], axis=1)

    # 3. Guardar el DataFrame final en el nuevo CSV
    df_final.to_csv(csv_path_output, index=False, encoding='utf-8-sig')

    # Cambiar el nombre de la columna generated_summary por MedGemma_pls
    df_final.rename(columns={'generated_summary': 'medgemma_pls'}, inplace=True)

    print("\n" + "="*80)
    print("¡Éxito! La generación de PLS ha finalizado.")
    print(f"Resultados guardados en: {csv_path_output}")
    print("Este archivo contiene las columnas 'technical_text', 'reference_summary', 'medgemma_pls', 'gemini_pls' y 'llama_pls'.")
    print("="*80)

    # 4. Mostrar un resumen de los resultados
    print("\nVista previa de los resultados (primeras 5 filas):")
    print(df_final.head())

    # 5. Comprobar si hubo errores finales
    errores_gemini = df_final[df_final['gemini_pls'].str.startswith("ERROR_GEMINI_FINAL")]
    errores_llama = df_final[df_final['llama_pls'].str.startswith("ERROR_LLAMA_FINAL")]

    if not errores_gemini.empty or not errores_llama.empty:
        print("\n" + "!"*80)
        print("ADVERTENCIA: Se detectaron errores finales después de 3 reintentos.")
        print(f"Total de errores finales en Gemini: {len(errores_gemini)}")
        print(f"Total de errores finales en Llama: {len(errores_llama)}")
        print("Estos necesitarán ser revisados o ejecutados manualmente.")
        print("!"*80)
    else:
        print("\nComprobación de integridad: No se detectaron errores finales. ¡Todo se generó exitosamente!")

except Exception as e:
    print(f"\nError al consolidar o guardar el DataFrame: {e}")
    print("Los resultados de la generación están en 'gemini_pls_list' y 'llama_pls_list'.")


¡Éxito! La generación de PLS ha finalizado.
Resultados guardados en: /content/drive/MyDrive/NLP_Project_MedGemma/v8_foundation_model_pls_results.csv
Este archivo contiene las columnas 'technical_text', 'reference_summary', 'medgemma_pls', 'gemini_pls' y 'llama_pls'.

Vista previa de los resultados (primeras 5 filas):
                                      technical_text  \
0  Background\nLumbar spinal stenosis with neurog...   
1  Background\nOlder patients with multiple healt...   
2  Background\nBeta‐blockers are an essential par...   
3  Background\nThalassaemia is a genetic disorder...   
4  Background\nApproximately 600 million children...   

                                   reference_summary  \
0  Non‐surgical treatment for spinal stenosis wit...   
1  Interventions for involving older patients wit...   
2  Beta‐blockers for children with congestive hea...   
3  Removal of the spleen in people with thalassae...   
4  One, two or three times a week iron supplement...   

      

In [ ]:
# --- Celda 1.1: Instalación de Dependencias (Sin AlignScore) ---

# Instalar librerías estándar desde PyPI
!pip install -q pandas torch transformers evaluate bert_score textstat spacy accelerate sentencepiece protobuf

# Descargar el modelo de lenguaje de spaCy
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.4/176.4 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 100.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 150.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## 8. Evaluación modular sin AlignScore
Prepara dependencias ligeras, carga modelos y evaluadores, ejecuta la lógica de evaluación y persiste métricas intermedias.

In [ ]:
# --- Celda 1.2: Importaciones y Configuración Inicial ---
import os
import pandas as pd
import numpy as np
import torch
import spacy
import textstat
import evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
from tqdm.auto import tqdm
import warnings
# limpia ram y vram
import gc
# Ignorar advertencias
warnings.filterwarnings("ignore")

In [ ]:
# Conectarse a Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# --- Configuración Global ---
BASE_PROJECT_DIR = "/content/drive/MyDrive/NLP_Project_MedGemma"
CSV_RESULTS_PATH = os.path.join(BASE_PROJECT_DIR, 'v8_foundation_model_pls_results.csv')
INTERMEDIATE_CSV_PATH = os.path.join(BASE_PROJECT_DIR, 'v8_intermediate_metrics.csv') # Archivo de salida

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {DEVICE}")

Usando dispositivo: cuda


In [ ]:
# --- Celda 1.3: Carga de Modelos y Evaluadores (Sin AlignScore) ---

print("Cargando modelo de Spacy...")
nlp = spacy.load("en_core_web_sm")

print("Cargando métrica BERTScore...")
bertscore_metric = evaluate.load("bertscore")

print("Inicializando modelo NLI para Factualidad...")
nli_model_id = "facebook/bart-large-mnli"
nli_tokenizer = AutoTokenizer.from_pretrained(nli_model_id)
nli_model = AutoModelForSequenceClassification.from_pretrained(nli_model_id).to(DEVICE)
nli_model.eval()

print("Inicializando pipelines de QA para Factualidad...")
question_generator = pipeline("text2text-generation", model="valhalla/t5-base-qg-hl", device=DEVICE)
question_answerer = pipeline("question-answering", model="distilbert-base-cased-distilled-squad", device=DEVICE)

print("\nModelos y evaluadores listos.")

Cargando modelo de Spacy...
Cargando métrica BERTScore...


Inicializando modelo NLI para Factualidad...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Device set to use cuda



Modelos y evaluadores listos.


In [ ]:
# --- Celda 1.4: Lógica de Evaluación Modular (Corregida y Robusta) ---

# Asegurarse de que el tokenizer tenga pad_token (BART a veces no lo trae por defecto en ciertas versiones)
if nli_tokenizer.pad_token is None:
    nli_tokenizer.pad_token = nli_tokenizer.eos_token
    print("Token de relleno (pad_token) asignado manualmente.")

# --- Función para BERTScore (Sin cambios) ---
def calculate_bertscore(predictions, references):
    if not predictions or not references:
        return 0.0
    print("Calculando BERTScore...")
    try:
        results = bertscore_metric.compute(predictions=predictions, references=references, lang="en", verbose=False)
        return np.mean(results['f1'])
    except Exception as e:
        print(f"Error en BERTScore: {e}")
        return 0.0

# --- Función para Factualidad (NLI) - CORREGIDA Y SEGURA ---
def calculate_nli_factuality(sources, predictions, batch_size=16):
    print("Calculando Factualidad (NLI) con procesamiento por lotes robusto...")

    # Listas separadas para premisas (texto fuente) e hipótesis (oración resumen)
    all_premises = []
    all_hypotheses = []
    doc_indices = [] # Para reconstruir a qué resumen pertenece cada par

    print("Tokenizando oraciones con Spacy...")
    # batch_size en nlp.pipe ayuda a gestionar memoria CPU
    docs = list(nlp.pipe(predictions, disable=["ner", "tagger", "lemmatizer"], batch_size=100))

    for i, (source, doc) in enumerate(zip(sources, docs)):
        # Validación extra: si source es vacío, lo saltamos para evitar errores del modelo
        if not isinstance(source, str) or not source.strip():
            continue

        summary_sentences = [sent.text for sent in doc.sents if sent.text.strip()]

        if not summary_sentences:
            continue

        for sent in summary_sentences:
            all_premises.append(source)
            all_hypotheses.append(sent)
            doc_indices.append(i)

    if not all_premises:
        return 0.0

    print(f"Ejecutando inferencia NLI en {len(all_premises)} pares...")
    probs_list = []

    # Procesamiento por lotes
    for i in tqdm(range(0, len(all_premises), batch_size), desc="NLI Batches"):
        batch_prem = all_premises[i : i + batch_size]
        batch_hypo = all_hypotheses[i : i + batch_size]

        # Tokenización: Pasamos (texto, par_texto) por separado
        inputs = nli_tokenizer(
            batch_prem,           # Premisa (Source)
            batch_hypo,           # Hipótesis (Summary sentence)
            return_tensors='pt',
            padding=True,         # Padding dinámico al más largo del batch
            truncation="only_first", # CRÍTICO: Solo recortar la fuente (premisa), nunca el resumen
            max_length=1024       # BART soporta hasta 1024. Si tienes poca VRAM, baja a 512
        ).to(DEVICE)

        with torch.no_grad():
            outputs = nli_model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1)

            # BART-Large-MNLI labels: 0: contradiction, 1: neutral, 2: entailment
            # Obtenemos probabilidad de contradicción (índice 0)
            contradiction_probs = probs[:, 0].cpu().numpy()
            probs_list.extend(contradiction_probs)

    # Reconstrucción de Scores
    summary_scores = {i: [] for i in range(len(predictions))}

    for idx, prob in zip(doc_indices, probs_list):
        # Umbral estricto
        is_contradiction = 1 if prob > 0.7 else 0
        summary_scores[idx].append(is_contradiction)

    final_scores = []
    for i in range(len(predictions)):
        sentences_contradictions = summary_scores[i]
        if not sentences_contradictions:
            final_scores.append(0.0)
        else:
            # Score = 1 - (% de oraciones que son mentira)
            score = 1.0 - (sum(sentences_contradictions) / len(sentences_contradictions))
            final_scores.append(score)
    # LIMPIEZA CRITICAL AL FINAL DE NLI
    del all_premises, all_hypotheses, doc_indices, probs_list, summary_scores
    gc.collect()
    torch.cuda.empty_cache()
    # Forzamos al recolector de basura de Python
    gc.collect()
    # Vaciamos la caché de PyTorch para liberar la VRAM
    torch.cuda.empty_cache()

    print("Memoria VRAM liberada tras QA.")

    return np.mean(final_scores)

# --- Función para Factualidad (QA Answerability) - VERSIÓN CON TQDM Y GESTIÓN DE VRAM ---
def calculate_qa_factuality(sources, predictions):
    print("Calculando Factualidad (QA Answerability) con procesamiento por lotes...")

    all_sentences = []
    contexts_for_sentences = []
    sentence_counts_per_summary = []

    # Uso de nlp.pipe para consistencia y velocidad
    docs = list(nlp.pipe(predictions, disable=["ner", "tagger"]))

    for source_doc, doc in zip(sources, docs):
        if not source_doc or not doc.text.strip():
            sentence_counts_per_summary.append(0)
            continue

        summary_sentences = [sent.text for sent in doc.sents if sent.text.strip()]

        if not summary_sentences:
            sentence_counts_per_summary.append(0)
            continue

        all_sentences.extend(summary_sentences)
        contexts_for_sentences.extend([source_doc] * len(summary_sentences))
        sentence_counts_per_summary.append(len(summary_sentences))

    if not all_sentences:
        print("No se encontraron oraciones válidas para procesar.")
        return 0.0

    print(f"Generando preguntas para {len(all_sentences)} oraciones en total...")
    # Generación de preguntas
    generated_q_dicts = question_generator(
        all_sentences,
        max_new_tokens=32,
        num_beams=4,
        batch_size=128, # Aumenté un poco el batch si la memoria lo permite
        truncation=True
    )
    questions = [q['generated_text'] for q in generated_q_dicts]

    print(f"Respondiendo a {len(questions)} preguntas...")
    answers = question_answerer(
        question=questions,
        context=contexts_for_sentences,
        batch_size=128
    )

    final_scores = []
    current_answer_idx = 0
    for num_sentences in tqdm(sentence_counts_per_summary, desc="Agregando scores"):
        if num_sentences == 0:
            final_scores.append(0)
            continue

        summary_answers = answers[current_answer_idx : current_answer_idx + num_sentences]
        answered_questions = sum(1 for ans in summary_answers if ans['score'] > 0.3) # Umbral de confianza
        score = answered_questions / num_sentences
        final_scores.append(score)
        current_answer_idx += num_sentences

    # --- 5. LIMPIEZA FINAL CRÍTICA ---
    # Borramos las variables grandes que ocupan memoria
    del all_sentences, contexts_for_sentences, questions, answers, docs
    # Forzamos al recolector de basura de Python
    gc.collect()
    # Vaciamos la caché de PyTorch para liberar la VRAM
    torch.cuda.empty_cache()

    print("Memoria VRAM liberada tras QA.")
    return np.mean(final_scores)

# --- Función para Métricas de Legibilidad (Sin cambios mayores, solo limpieza) ---
def calculate_readability(texts):
    print("Calculando Métricas de Legibilidad...")
    metrics = {'fkgl': [], 'fre': [], 'fog': [], 'smog': [], 'cl': []}
    for text in tqdm(texts, desc="Readability"):
         if not isinstance(text, str) or not text.strip():
            metrics['fkgl'].append(12.0); metrics['fre'].append(30.0); metrics['fog'].append(12.0); metrics['smog'].append(8.0); metrics['cl'].append(8.0)
            continue
         metrics['fkgl'].append(textstat.flesch_kincaid_grade(text))
         metrics['fre'].append(textstat.flesch_reading_ease(text))
         metrics['fog'].append(textstat.gunning_fog(text))
         metrics['smog'].append(textstat.smog_index(text))
         metrics['cl'].append(textstat.coleman_liau_index(text))

    return {key: np.mean(val) for key, val in metrics.items()}

In [ ]:
gc.collect()
torch.cuda.empty_cache()
## Forzamos al recolector de basura de Python
#gc.collect()
## Vaciamos la caché de PyTorch para liberar la VRAM
#torch.cuda.empty_cache()

In [ ]:
# --- Celda 1.5: Ejecución y Almacenamiento de Resultados Intermedios ---

# Cargar datos
print(f"Cargando datos desde: {CSV_RESULTS_PATH}")
df_results = pd.read_csv(CSV_RESULTS_PATH)
df_results.dropna(subset=['technical_text', 'reference_summary'], inplace=True)

# For testing: Filtrar solo 10 textos
#df_results = df_results.head(10)

if 'generated_summary' in df_results.columns:
    df_results.rename(columns={'generated_summary': 'medgemma_pls'}, inplace=True)

# Limpieza inicial general para asegurar que references/sources sean válidos
df_results = df_results[df_results['reference_summary'].str.strip() != '']
df_results = df_results[df_results['technical_text'].str.strip() != '']

model_columns = [col for col in df_results.columns if '_pls' in col]
results_list = []

Cargando datos desde: /content/drive/MyDrive/NLP_Project_MedGemma/v8_foundation_model_pls_results.csv


In [ ]:
%%time
# ---------------------------------------------------------
# NUEVO BLOQUE: Evaluación del Reference Summary (Baseline)
# ---------------------------------------------------------
print(f"\n{'='*25}\n--- Evaluando Referencia: reference_summary ---\n{'='*25}")

ref_sources = df_results['technical_text'].tolist()
ref_texts = df_results['reference_summary'].tolist()

# 1. Readability (Referencia)
ref_readability = calculate_readability(ref_texts)
for key, value in ref_readability.items():
    results_list.append({'model': 'reference_summary', 'metric': f'readability_{key}', 'score': value})

# 2. BERTScore (Referencia vs Referencia es siempre 1.0)
# Lo agregamos hardcodeado para que aparezca en las gráficas comparativas
results_list.append({'model': 'reference_summary', 'metric': 'bertscore_f1', 'score': 1.0})

# 3. Factualidad NLI (Referencia vs Source)
# Esto nos dice si el humano fue fiel al texto fuente
ref_nli_score = calculate_nli_factuality(ref_sources, ref_texts)
results_list.append({'model': 'reference_summary', 'metric': 'nli_factuality', 'score': ref_nli_score})

# 4. Factualidad QA (Referencia vs Source)
ref_qa_score = calculate_qa_factuality(ref_sources, ref_texts)
results_list.append({'model': 'reference_summary', 'metric': 'qa_factuality', 'score': ref_qa_score})

# ---------------------------------------------------------
# Evaluación de Modelos
# ---------------------------------------------------------
for model_name in model_columns:
    print(f"\n{'='*25}\n--- Evaluando Modelo: {model_name} ---\n{'='*25}")

    # Filtrar filas donde este modelo específico tenga predicciones nulas
    temp_df = df_results[['technical_text', 'reference_summary', model_name]].copy()
    temp_df.dropna(subset=[model_name], inplace=True)
    temp_df = temp_df[temp_df[model_name].str.strip() != '']

    if temp_df.empty:
        print(f"No hay resúmenes válidos para {model_name}. Saltando.")
        continue

    sources = temp_df['technical_text'].tolist()
    references = temp_df['reference_summary'].tolist()
    predictions = temp_df[model_name].tolist()

    # Calcular métricas
    # BERTScore
    results_list.append({'model': model_name, 'metric': 'bertscore_f1', 'score': calculate_bertscore(predictions, references)})

    # Factualidad (NLI & QA)
    results_list.append({'model': model_name, 'metric': 'nli_factuality', 'score': calculate_nli_factuality(sources, predictions)})
    results_list.append({'model': model_name, 'metric': 'qa_factuality', 'score': calculate_qa_factuality(sources, predictions)})

    # Readability
    readability_scores = calculate_readability(predictions)
    for key, value in readability_scores.items():
        results_list.append({'model': model_name, 'metric': f'readability_{key}', 'score': value})

    # --- AGREGAR ESTO AL FINAL DEL BUCLE FOR ---
    print(f"Limpiando memoria tras evaluar {model_name}...")
    del sources, references, predictions, temp_df
    gc.collect()
    torch.cuda.empty_cache()
    print("-" * 30)

# Convertir y guardar
intermediate_df = pd.DataFrame(results_list)
intermediate_df.to_csv(INTERMEDIATE_CSV_PATH, index=False)

print("\n\n" + "="*80)
print("¡Éxito! Las métricas (incluyendo baseline de referencia) han sido calculadas.")
print(f" --> {INTERMEDIATE_CSV_PATH}")
print("="*80)
print(intermediate_df.head(10))


--- Evaluando Referencia: reference_summary ---
Calculando Métricas de Legibilidad...


Readability:   0%|          | 0/100 [00:00<?, ?it/s]

Calculando Factualidad (NLI) con procesamiento por lotes robusto...
Tokenizando oraciones con Spacy...
Ejecutando inferencia NLI en 2083 pares...


NLI Batches:   0%|          | 0/131 [00:00<?, ?it/s]

Memoria VRAM liberada tras QA.
Calculando Factualidad (QA Answerability) con procesamiento por lotes...
Generando preguntas para 2083 oraciones en total...
Respondiendo a 2083 preguntas...


...

Memoria VRAM liberada tras QA.
Calculando Factualidad (QA Answerability) con procesamiento por lotes...
Generando preguntas para 1594 oraciones en total...
Respondiendo a 1594 preguntas...


Agregando scores:   0%|          | 0/100 [00:00<?, ?it/s]

Memoria VRAM liberada tras QA.
Calculando Métricas de Legibilidad...


Readability:   0%|          | 0/100 [00:00<?, ?it/s]

Limpiando memoria tras evaluar llama_pls...
------------------------------


¡Éxito! Las métricas (incluyendo baseline de referencia) han sido calculadas.
 --> /content/drive/MyDrive/NLP_Project_MedGemma/v8_intermediate_metrics.csv
               model            metric      score
0  reference_summary  readability_fkgl  13.963180
1  reference_summary   readability_fre  34.247340
2  reference_summary   readability_fog  16.915162
3  reference_summary  readability_smog  15.456659
4  reference_summary    readability_cl  14.118936
5  reference_summary      bertscore_f1   1.000000
6  reference_summary    nli_factuality   0.945061
7  reference_summary     qa_factuality   0.163167
8       medgemma_pls      bertscore_f1   0.852164
9       medgemma_pls    nli_factuality   0.976970
CPU times: user 14min 41s, sys: 9.87 s, total: 14min 51s
Wall time: 14min 18s


## 9. Flujo AlignScore en entorno aislado
Instala Python 3.10 en un entorno virtual, agrega AlignScore y spaCy, escribe el script de puntuación y lo ejecuta para obtener el reporte final.

In [ ]:
# --- Celda 2.1: Instalar Python 3.10 ---
# Añadimos el repositorio PPA 'deadsnakes' que contiene versiones antiguas de Python
!sudo add-apt-repository -y ppa:deadsnakes/ppa
!sudo apt-get update -y

# Instalamos Python 3.10 y su módulo venv
!sudo apt-get install -qq python3.10 python3.10-venv -y

# Verificamos que se instaló correctamente
!python3.10 --version

Repository: 'deb https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/ jammy main'
Description:
This PPA contains more recent Python versions packaged for Ubuntu.

Disclaimer: there's no guarantee of timely updates in case of security problems or other issues. If you want to use them in a security-or-otherwise-critical environment (say, on a production server), you do so at your own risk.

Update Note
Please use this repository instead of ppa:fkrull/deadsnakes.

Reporting Issues

Issues can be reported in the master issue tracker at:
https://github.com/deadsnakes/issues/issues

Supported Ubuntu and Python Versions

- Ubuntu 22.04 (jammy) Python3.7 - Python3.9, Python3.11 - Python3.13
- Ubuntu 24.04 (noble) Python3.7 - Python3.11, Python3.13
- Note: Python 3.10 (jammy), Python3.12 (noble) are not provided by deadsnakes as upstream ubuntu provides those packages.

Why some packages aren't built:
- Note: for jammy and noble, older python versions requre libssl<3 so they are not currentl

In [ ]:
# --- Celda 2.2: Crear y Preparar el Entorno Virtual (VERSIÓN FINAL CORREGIDA) ---

# 1. Creamos un entorno virtual (esto no cambia)
!python3.10 -m venv /content/align_env

# 2. Instalamos PyTorch (esto no cambia)
!/content/align_env/bin/pip install -q torch==1.13.1+cu117 torchvision==0.14.1+cu117 --extra-index-url https://download.pytorch.org/whl/cu117

# 3. ¡EL COMANDO CLAVE Y ÚNICO!
#    Instalamos AlignScore Y TODAS sus dependencias con las versiones correctas en un solo paso.
#    - transformers==4.33.2 (para el import AdamW)
#    - tokenizers==0.13.3 (compatible con transformers 4.33.2)
#    - pytorch-lightning==1.9.5 (la versión que AlignScore espera)
!/content/align_env/bin/pip install -q \
    pandas \
    transformers==4.33.2 \
    tokenizers==0.13.3 \
    pytorch-lightning==1.9.5 \
    "git+https://github.com/yuh-zha/AlignScore.git"

# --- Verificación ---
print("\n--- Verificando el entorno virtual ---")
!/content/align_env/bin/python --version
!/content/align_env/bin/python -c "import torch; print(f'PyTorch version: {torch.__version__}'); import transformers; print(f'Transformers version: {transformers.__version__}'); import alignscore; print('¡ÉXITO! AlignScore y sus dependencias se importaron correctamente.')"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 GB 888.3 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.3/24.3 MB 80.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 KB 7.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 104.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 KB 9.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 116.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.8/129.8 KB 20.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.6/153.6 KB 24.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.4/159.4 KB 25.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 KB 11.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 1

In [ ]:
# --- Celda 2.3: Instalar AlignScore y Descargar el Modelo ---

# Usamos el pip que está DENTRO de nuestro entorno conda para instalar AlignScore
!/usr/local/envs/alignscore_env/bin/pip install -q pandas git+https://github.com/yuh-zha/AlignScore.git

# Clonar el repositorio para tener una estructura de carpetas predecible
!git clone https://github.com/yuh-zha/AlignScore.git

# Descargar el checkpoint del modelo en el directorio clonado
!cd /content/AlignScore/ && wget https://huggingface.co/yzha/AlignScore/resolve/main/AlignScore-base.ckpt

# Verificar que el archivo se descargó
!echo -e "\n--- Verificación de Archivos ---"
!ls -lh /content/AlignScore/

/bin/bash: line 1: /usr/local/envs/alignscore_env/bin/pip: No such file or directory
Cloning into 'AlignScore'...
remote: Enumerating objects: 202, done.
remote: Counting objects: 100% (49/49), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 202 (delta 36), reused 12 (delta 12), pack-reused 153 (from 1)
Receiving objects: 100% (202/202), 523.86 KiB | 14.55 MiB/s, done.
Resolving deltas: 100% (113/113), done.
--2025-11-20 17:51:08--  https://huggingface.co/yzha/AlignScore/resolve/main/AlignScore-base.ckpt
Resolving huggingface.co (huggingface.co)... 13.35.202.34, 13.35.202.121, 13.35.202.40, ...
Connecting to huggingface.co (huggingface.co)|13.35.202.34|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/647835e2159a889d001bebb4/a1ae41aa60f0f09954a2e05d43602df4f5edd951c2d85092bfc0002bc9a3f8c8?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27AlignScore-base.ckpt%3B+filename%3D%22AlignScore

In [ ]:
# 4. ¡NUEVO PASO! Descargamos el modelo de spaCy USANDO el python del entorno virtual.
#    Esto asegura que el modelo sea visible para el script.
!/content/align_env/bin/python -m spacy download en_core_web_sm

# --- Verificación ---
print("\n--- Verificando el entorno virtual ---")
!/content/align_env/bin/python --version
# La verificación ahora también comprueba que spacy puede cargar el modelo.
!/content/align_env/bin/python -c "import torch; print(f'PyTorch version: {torch.__version__}'); import transformers; print(f'Transformers version: {transformers.__version__}'); import alignscore; import spacy; nlp=spacy.load('en_core_web_sm'); print('¡ÉXITO! Todas las dependencias, incluyendo spaCy, se importaron y cargaron correctamente.')"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 106.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')

--- Verificando el entorno virtual ---
Python 3.10.12
PyTorch version: 1.13.1+cu117
Transformers version: 4.33.2
¡ÉXITO! Todas las dependencias, incluyendo spaCy, se importaron y cargaron correctamente.


In [ ]:
# --- Celda 2.4: Escribir el Script de Ejecución (VERSIÓN OPTIMIZADA) ---

# 2. Escribir todo nuestro código de Python en un archivo para ejecutarlo con el venv.
%%writefile run_alignscore_evaluation.py

# Se importan todas las librerías necesarias DENTRO del script
import os
import pandas as pd
import numpy as np
import torch
from tqdm.auto import tqdm
from alignscore import AlignScore
import nltk
nltk.download('punkt_tab')

print("--- Script de evaluación optimizado iniciado ---")

# --- Configuración Global ---
BASE_PROJECT_DIR = "/content/drive/MyDrive/NLP_Project_MedGemma"
BASE_ALIGNSCORE_DIR = "/content/AlignScore"

CSV_RESULTS_PATH = os.path.join(BASE_PROJECT_DIR, 'v8_foundation_model_pls_results.csv')
INTERMEDIATE_CSV_PATH = os.path.join(BASE_PROJECT_DIR, 'v8_intermediate_metrics.csv')
FINAL_CSV_PATH = os.path.join(BASE_PROJECT_DIR, 'v8_final_combined_metrics.csv')
ALIGN_SCORE_CKPT_PATH = os.path.join(BASE_ALIGNSCORE_DIR, 'AlignScore-base.ckpt')

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando dispositivo: {DEVICE}")
print(f"Versión de PyTorch detectada: {torch.__version__}")

# --- Cargar AlignScore ---
print("\nInicializando AlignScore...")
# Aumenta el batch_size si tienes una GPU potente como la A100 (ej. 32 o 64)
alignscore_evaluator = AlignScore(
    model='roberta-large', batch_size=64, device=DEVICE,
    ckpt_path=ALIGN_SCORE_CKPT_PATH, evaluation_mode='nli_sp'
)
print("AlignScore cargado exitosamente.")

# --- Lógica de Cálculo Optimizada ---
try:
    df_original = pd.read_csv(CSV_RESULTS_PATH)
    df_original.dropna(subset=['technical_text', 'reference_summary'], inplace=True)

    # Renombrar generated_summary por medgemma_pls
    if 'generated_summary' in df_original.columns:
      df_original.rename(columns={'generated_summary': 'medgemma_pls'}, inplace=True)

    model_columns = [col for col in df_original.columns if '_pls' in col]
    df_intermediate = pd.read_csv(INTERMEDIATE_CSV_PATH)

    # Renombrar generated_summary por medgemma_pls
    if 'generated_summary' in df_intermediate.columns:
        df_intermediate.rename(columns={'generated_summary': 'medgemma_pls'}, inplace=True)

    print(f"\nModelos a procesar: {model_columns}")

    # --- 1. Preparación: Recolectar TODOS los datos de TODOS los modelos ---
    print("\nPreparando datos de todos los modelos para un único procesamiento...")
    all_sources = []
    all_predictions = []
    prediction_counts_per_model = {} # Para reconstruir los scores

    for model_name in tqdm(model_columns, desc="Recolectando predicciones"):
        temp_df = df_original[['technical_text', model_name]].copy()
        temp_df.dropna(subset=[model_name], inplace=True)
        temp_df = temp_df[temp_df[model_name].str.strip() != '']

        if temp_df.empty:
            prediction_counts_per_model[model_name] = 0
            continue

        sources = temp_df['technical_text'].tolist()
        predictions = temp_df[model_name].tolist()

        all_sources.extend(sources)
        all_predictions.extend(predictions)
        prediction_counts_per_model[model_name] = len(predictions)

    # --- 2. Ejecución: Calcular TODOS los scores en una única llamada masiva ---
    if not all_predictions:
        print("No se encontraron predicciones válidas en ningún modelo.")
        all_scores = []
    else:
        print(f"\nCalculando AlignScore para {len(all_predictions)} pares (fuente, predicción) en total...")
        # La librería AlignScore se encarga internamente del batching de esta gran lista
        all_scores = alignscore_evaluator.score(contexts=all_sources, claims=all_predictions)

    # --- 3. Re-agregación: Distribuir los scores de vuelta a cada modelo ---
    print("\nAgregando y asignando scores a sus respectivos modelos...")
    alignscore_results = []
    current_score_idx = 0
    for model_name in model_columns:
        count = prediction_counts_per_model.get(model_name, 0)

        if count == 0:
            # Asignar un score de 0 si el modelo no tuvo predicciones válidas
            score = 0.0
        else:
            model_scores = all_scores[current_score_idx : current_score_idx + count]
            score = np.mean(model_scores)
            current_score_idx += count # Mover el puntero

        alignscore_results.append({'model': model_name, 'metric': 'alignscore', 'score': score})
        print(f"Score final para {model_name}: {score:.4f}")

    # --- 4. Combinar y Guardar Resultados ---
    df_alignscore = pd.DataFrame(alignscore_results)
    df_final_metrics = pd.concat([df_intermediate, df_alignscore], ignore_index=True)

    df_final_metrics.to_csv(FINAL_CSV_PATH, index=False)
    print(f"\n¡Éxito! Resultados combinados guardados en: {FINAL_CSV_PATH}")

except FileNotFoundError as e:
    print(f"\nERROR: No se pudo encontrar un archivo necesario: {e}")
except Exception as e:
    print(f"\nOcurrió un error inesperado durante la ejecución: {e}")
finally:
    print("--- Script de evaluación finalizado ---")

Overwriting run_alignscore_evaluation.py


In [ ]:
# --- Celda 2.5: Ejecutar el Script y Generar el Reporte Final ---

# 1. Ejecutar el script completo usando el intérprete de Python de nuestro entorno virtual.
#    Toda la lógica que depende de AlignScore ocurre aquí.
print("="*80)
print("Iniciando la ejecución del script de evaluación en el entorno virtual...")
print("="*80)
!/content/align_env/bin/python run_alignscore_evaluation.py
print("="*80)
print("...Ejecución del script finalizada.")
print("="*80)


# 2. Ahora, de vuelta en el kernel de Colab, cargamos el archivo de resultados
#    que el script acaba de crear.
FINAL_CSV_PATH = "/content/drive/MyDrive/NLP_Project_MedGemma/v8_final_combined_metrics.csv"
try:
    final_pivot = pd.read_csv(FINAL_CSV_PATH).pivot(index='metric', columns='model', values='score')

    # 3. Generar la tabla de reporte final (esto se ejecuta en el kernel de Colab)
    metric_map = {
        'bertscore_f1': 'BERTScore - F1', 'alignscore': 'AlignScore (NLI_SP)',
        'nli_factuality': 'Factualidad (1 - Contradiction Ratio)', 'qa_factuality': 'Factualidad (QA Answerability)',
        'readability_fkgl': 'Flesch-Kincaid Grade', 'readability_fre': 'Flesch Reading Ease',
        'readability_fog': 'Gunning Fog Index', 'readability_smog': 'SMOG Index',
        'readability_cl': 'Coleman-Liau Index',
    }
    metric_order = [
        'bertscore_f1', 'alignscore', 'nli_factuality', 'qa_factuality', '---',
        'readability_fkgl', 'readability_fre', 'readability_fog', 'readability_smog', 'readability_cl'
    ]
    targets = ['≥ 0.86', '≥ 0.80', '≥ 0.90', '≥ 0.80', '', '≤ 8.0', '≥ 60', '≤ 12.0', '≤ 8.0', '≤ 8.0']

    report_df = pd.DataFrame({'Métrica': [metric_map.get(m, '---') for m in metric_order], 'Meta / Objetivo': targets})
    def get_score(metric, model):
        try: return final_pivot.loc[metric, model]
        except KeyError: return 'N/A'

    report_df['PLS Reales (Referencia)'] = [
        'N/A', 'N/A', 'N/A', 'N/A', '',
        f"{get_score('readability_fkgl', 'reference_summary'):.2f}", f"{get_score('readability_fre', 'reference_summary'):.2f}",
        f"{get_score('readability_fog', 'reference_summary'):.2f}", f"{get_score('readability_smog', 'reference_summary'):.2f}",
        f"{get_score('readability_cl', 'reference_summary'):.2f}",
    ]
    model_columns = sorted([col for col in final_pivot.columns if col != 'reference_summary'])
    for model_name in model_columns:
        report_df[model_name] = [
            f"{get_score('bertscore_f1', model_name):.4f}", f"{get_score('alignscore', model_name):.4f}",
            f"{get_score('nli_factuality', model_name):.4f}", f"{get_score('qa_factuality', model_name):.4f}", '',
            f"{get_score('readability_fkgl', model_name):.2f}", f"{get_score('readability_fre', model_name):.2f}",
            f"{get_score('readability_fog', model_name):.2f}", f"{get_score('readability_smog', model_name):.2f}",
            f"{get_score('readability_cl', model_name):.2f}",
        ]

    # --- Mostrar la Tabla Final ---
    print("\n\n" + "="*100)
    print("--- Tabla Comparativa Final de Resultados de Evaluación (Combinada) ---")
    print("="*100)
    print(report_df.to_string(index=False))
    print("="*100)

except FileNotFoundError:
    print(f"\nERROR CRÍTICO: No se encontró el archivo de salida en {FINAL_CSV_PATH}.")
    print("Revisa la salida del script anterior para ver si hubo errores durante su ejecución.")

Iniciando la ejecución del script de evaluación en el entorno virtual...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
--- Script de evaluación optimizado iniciado ---
Usando dispositivo: cuda
Versión de PyTorch detectada: 1.13.1+cu117

Inicializando AlignScore...
/content/align_env/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.weight', 'roberta.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Lightning automatically upgraded your loaded checkpoint from v1.7.7 to v1.9.5. 